# Classificação de Intenções — Agente de IA para o Portfólio de Projetos do Metrô (Grupo Oneshot / g05)

Notebook de ponta a ponta para o núcleo de PLN do MVP (RF01–RF03 e RF07):

1. **Catálogo de intenções** em dois níveis — *macrointenção* (`CONSULTA`, `TRANSACAO`, `ALERTA`, conforme o enum `CategoriaIntencao` do `Projeto.md`, mais `FORA_DE_ESCOPO` para rejeição) e *intenção específica* (31 classes).
2. **Dicionário extenso de prompts** gerado por *templates + slots* do domínio (projetos, subportfólios, marcos, orçamento, riscos, TAP, indicadores).
3. **Split sem vazamento**: a divisão treino/validação/teste é feita **por template (grupo)**, nunca por frase. Nenhum template visto no treino aparece na validação ou no teste, e há deduplicação de textos normalizados entre splits.
4. **Seleção de pré-processamento** entre 36 configurações, com `StratifiedGroupKFold` aplicado **somente ao treino**; na execução de referência, venceu a `config_22`.
5. **Benchmark de representações** lexicais e semânticas; na execução completa de referência, a melhor representação foi o **híbrido Word2Vec + BERT**.
6. **Classificação hierárquica** (macro → específica) comparada ao modelo plano, refinamento de hiperparâmetros, limiar de rejeição e avaliação final no teste.
---

### Rastreabilidade com os requisitos

| Seção do notebook | Requisito |
|---|---|
| Catálogo de intenções | RF02, RF03 |
| Dicionário de prompts | RF01 |
| Modelos e avaliação | RF02, RF03 |
| Roteamento por macrointenção (`classificar`) | RF07 |
| Limiar de rejeição / "não sei" | RF08 (indicar limitação) |

### Resultado principal da execução de referência

O benchmark selecionou o **Híbrido W2V+BERT**, com classificação **hierárquica** e regressão logística (`C=5.0`, `class_weight=balanced`). A combinação aproveita a semântica distribucional aprendida pelo Word2Vec no vocabulário do domínio e o contexto capturado pelos embeddings BERT congelados.

| Conjunto | Acurácia da intenção | F1-macro da intenção | Acurácia da macro | F1-macro da macro |
|---|---:|---:|---:|---:|
| Validação | 0,5666 | 0,5665 | 0,9083 | 0,8885 |
| Teste | **0,6740** | **0,6777** | **0,9383** | **0,9379** |

> Esses valores identificam a configuração atualmente escolhida pelo grupo; não tornam o modelo definitivo. A seção 9 discute a diferença entre validação e teste, os dados sintéticos e os limiares conservadores.

In [1]:
# ============================================================
# 0.1 Imports, seed e utilitários
# ============================================================
import os, re, json, math, random, string, unicodedata, warnings
from itertools import product
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.tokenize import word_tokenize, wordpunct_tokenize, WhitespaceTokenizer, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, RSLPStemmer, WordNetLemmatizer
from nltk.util import ngrams

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, PredefinedSplit
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

for pkg in ["punkt", "punkt_tab", "stopwords", "rslp"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print("Falha ao baixar", pkg, e)

try:
    import gensim
    from gensim.models import Word2Vec
    W2V_OK = True
except Exception as e:
    W2V_OK = False
    print("gensim indisponível:", e)

print("numpy", np.__version__, "| pandas", pd.__version__, "| gensim ok:", W2V_OK)

Matplotlib is building the font cache; this may take a moment.


numpy 1.26.4 | pandas 2.3.3 | gensim ok: True


## 1. Catálogo de intenções

Dois níveis, exatamente como previsto na modelagem estática do `Projeto.md`:

- **Macrointenção** (`CategoriaIntencao`): determina o **fluxo de processamento** (RF07).
  - `CONSULTA` — recuperação e análise de informações do portfólio (RF04, RF05, RF06).
  - `TRANSACAO` — execução/simulação de ações sobre os artefatos de gestão.
  - `ALERTA` — monitoramento proativo e notificações.
  - `FORA_DE_ESCOPO` — classe de rejeição (extensão operacional ao enum): sauda­ções, conversa social e domínios externos ao portfólio. Evita que o agente force uma das três categorias válidas e sustenta o RF08 ("indicar quando não houver informações suficientes").
- **Intenção específica**: 31 classes, incluindo as duas classes de fora de escopo; as intenções acionáveis possuem *handler* previsto no pipeline.

In [2]:
# ============================================================
# 1. Catálogo de intenções (macrointenção -> intenção específica)
# ============================================================

CATALOGO = {
    "CONSULTA": {
        "descricao": "Recuperação e análise de informações do portfólio (RF04/RF05/RF06).",
        "fluxo": "RecuperadorInformacao -> Estruturador de Resposta",
        "intencoes": {
            "consulta_status_projeto":       "Situação/andamento atual de um projeto ou programa.",
            "consulta_cronograma_prazo":     "Datas, prazos, previsão de término, atraso em dias.",
            "consulta_marcos_entregas":      "Marcos, entregas e etapas planejadas ou concluídas.",
            "consulta_orcamento_custo":      "Orçamento aprovado, custo realizado, saldo, desvio financeiro.",
            "consulta_escopo_tap":           "Escopo, objetivos, premissas e restrições registrados no TAP.",
            "consulta_responsavel_equipe":   "Gerente responsável, equipe, área ou patrocinador do projeto.",
            "consulta_indicadores_kpi":      "Indicadores de desempenho (SPI, CPI, % concluído, aderência).",
            "consulta_riscos_projeto":       "Riscos registrados, probabilidade, impacto e tratamento.",
            "consulta_pendencias_projeto":   "Pendências, ações em aberto e itens aguardando resposta.",
            "consulta_documentos_fonte":     "Localização/origem de documentos e referências (SharePoint, anexos).",
            "consulta_visao_portfolio":      "Visão agregada do portfólio ou de um subportfólio.",
            "comparacao_entre_projetos":     "Comparação explícita entre dois ou mais projetos (RF06).",
            "analise_tendencia_desempenho":  "Evolução temporal, tendência, projeção e causas de variação.",
        },
    },
    "TRANSACAO": {
        "descricao": "Execução ou simulação de ações sobre os artefatos de gestão (RF07).",
        "fluxo": "Executor de Transação -> Confirmação -> Registro",
        "intencoes": {
            "transacao_atualizar_status":        "Atualizar status/percentual de andamento de um projeto.",
            "transacao_atualizar_cronograma":    "Repactuar datas, replanejar marcos, prorrogar prazo.",
            "transacao_registrar_risco":         "Registrar/atualizar risco na matriz do projeto.",
            "transacao_registrar_pendencia":     "Abrir, atribuir ou encerrar pendência/ação.",
            "transacao_gerar_relatorio":         "Gerar relatório (anual, de acompanhamento, balanço).",
            "transacao_gerar_apresentacao":      "Gerar apresentação executiva a partir de template.",
            "transacao_exportar_dados":          "Exportar dados/planilha do portfólio.",
            "transacao_agendar_compromisso":     "Agendar reunião, checkpoint ou revisão de portfólio.",
            "transacao_solicitar_atualizacao":   "Solicitar/cobrar atualização de informação a um responsável.",
            "transacao_cadastrar_projeto":       "Cadastrar novo projeto/programa no portfólio.",
        },
    },
    "ALERTA": {
        "descricao": "Monitoramento proativo e notificações (RF07).",
        "fluxo": "Motor de Regras -> Notificação",
        "intencoes": {
            "alerta_prazo_atraso":            "Avisar sobre atraso ou risco de estouro de prazo.",
            "alerta_estouro_orcamento":       "Avisar sobre desvio ou estouro orçamentário.",
            "alerta_risco_critico":           "Avisar quando risco crítico for identificado/escalado.",
            "alerta_desatualizacao_dados":    "Avisar quando informações do projeto ficarem desatualizadas.",
            "alerta_configurar_monitoramento":"Criar/configurar regra de monitoramento e frequência.",
            "alerta_gerenciar_inscricao":     "Ativar, pausar, silenciar ou cancelar alertas existentes.",
        },
    },
    "FORA_DE_ESCOPO": {
        "descricao": "Rejeição: fora do domínio do portfólio (extensão operacional ao enum).",
        "fluxo": "Resposta padrão de limitação (RF08)",
        "intencoes": {
            "fora_escopo_social":   "Saudações, agradecimentos, conversa social, testes do agente.",
            "fora_escopo_externo":  "Assuntos alheios ao portfólio (clima, esportes, receitas, RH pessoal).",
        },
    },
}

# Mapas auxiliares
INTENCAO2MACRO = {i: m for m, bloco in CATALOGO.items() for i in bloco["intencoes"]}
MACROS = list(CATALOGO.keys())
INTENCOES = list(INTENCAO2MACRO.keys())

df_catalogo = pd.DataFrame(
    [{"macrointencao": m, "intencao": i, "descricao": d, "fluxo": CATALOGO[m]["fluxo"]}
     for m, bloco in CATALOGO.items() for i, d in bloco["intencoes"].items()]
)

print(f"{len(MACROS)} macrointenções | {len(INTENCOES)} intenções específicas")
display(df_catalogo)

4 macrointenções | 31 intenções específicas


,macrointencao,intencao,descricao,fluxo
0,CONSULTA,consulta_status_projeto,Situação/andamento atual de um projeto ou prog...,RecuperadorInformacao -> Estruturador de Resposta
1,CONSULTA,consulta_cronograma_prazo,"Datas, prazos, previsão de término, atraso em ...",RecuperadorInformacao -> Estruturador de Resposta
2,CONSULTA,consulta_marcos_entregas,"Marcos, entregas e etapas planejadas ou conclu...",RecuperadorInformacao -> Estruturador de Resposta
3,CONSULTA,consulta_orcamento_custo,"Orçamento aprovado, custo realizado, saldo, de...",RecuperadorInformacao -> Estruturador de Resposta
4,CONSULTA,consulta_escopo_tap,"Escopo, objetivos, premissas e restrições regi...",RecuperadorInformacao -> Estruturador de Resposta
5,CONSULTA,consulta_responsavel_equipe,"Gerente responsável, equipe, área ou patrocina...",RecuperadorInformacao -> Estruturador de Resposta
6,CONSULTA,consulta_indicadores_kpi,"Indicadores de desempenho (SPI, CPI, % concluí...",RecuperadorInformacao -> Estruturador de Resposta
7,CONSULTA,consulta_riscos_projeto,"Riscos registrados, probabilidade, impacto e t...",RecuperadorInformacao -> Estruturador de Resposta
8,CONSULTA,consulta_pendencias_projeto,"Pendências, ações em aberto e itens aguardando...",RecuperadorInformacao -> Estruturador de Resposta
9,CONSULTA,consulta_documentos_fonte,Localização/origem de documentos e referências...,RecuperadorInformacao -> Estruturador de Resposta


## 2. Dicionário extenso de prompts

Os prompts são gerados por **templates com slots** preenchidos com vocabulário real do domínio (projetos e programas do Metrô, subportfólios, indicadores, documentos, responsáveis, períodos).

Duas decisões importantes para a integridade do experimento:

- Cada frase carrega o **`template_id` que a originou**. Esse identificador é o *grupo* usado no split — é o que impede que uma paráfrase quase idêntica apareça no treino e no teste (o modo mais comum de vazamento em datasets sintéticos de intenção).
- A variação superficial (com/sem acento, com/sem interrogação, abreviações, caixa) é aplicada **dentro do grupo**, nunca criando um novo grupo, para não fabricar independência que não existe.

In [3]:
# ============================================================
# 2.1 Slots — vocabulário do domínio (Metrô de São Paulo / PMO)
# ============================================================

SLOTS = {
    "proj": [
        "Linha 2-Verde", "Linha 4-Amarela", "extensão da Linha 5-Lilás",
        "modernização da Linha 3-Vermelha", "Linha 15-Prata", "Linha 17-Ouro",
        "Pátio Oratório", "Estação Vila Sônia", "Estação Santa Marina",
        "programa de acessibilidade", "programa de eficiência energética",
        "projeto de renovação de frota", "projeto de sinalização CBTC",
        "projeto XPTO-104", "PRJ-2031", "PRJ-1187", "PRJ-0442",
        "modernização do CCO", "implantação do portal do passageiro",
        "programa de segurança patrimonial", "projeto de telecomunicações",
        "reforma da Estação Santana", "projeto de bilhetagem eletrônica",
        "programa de manutenção preditiva",
    ],
    "sub": [
        "subportfólio de Expansão", "subportfólio de Modernização",
        "subportfólio de Manutenção", "subportfólio de Tecnologia da Informação",
        "subportfólio de Sustentabilidade", "subportfólio de Segurança Operacional",
        "subportfólio de Acessibilidade", "carteira de obras civis",
    ],
    "per": [
        "neste mês", "no último trimestre", "em 2026", "no acumulado do ano",
        "na semana passada", "desde janeiro", "nos últimos seis meses",
        "no primeiro semestre", "no exercício atual",
    ],
    "resp": [
        "Roberto", "Mariana", "Carlos", "o gerente do projeto", "o PMO",
        "a Gerência de Expansão", "a GTI", "a equipe de engenharia",
        "o patrocinador", "a diretoria de operações",
    ],
    "doc": [
        "TAP", "Termo de Abertura do Projeto", "relatório anual de fechamento",
        "balanço do portfólio", "cronograma físico-financeiro", "ata da última reunião",
        "planilha de marcos", "relatório de acompanhamento", "matriz de riscos",
    ],
    "crit": [
        "prazo", "custo", "avanço físico", "nível de risco",
        "aderência ao cronograma", "execução orçamentária", "criticidade",
        "quantidade de pendências",
    ],
    "ind": [
        "SPI", "CPI", "percentual de conclusão", "aderência ao cronograma",
        "índice de desvio de custo", "curva S", "índice de execução orçamentária",
    ],
    "freq": [
        "diariamente", "toda segunda-feira", "semanalmente", "todo dia 1º",
        "mensalmente", "a cada quinze dias", "toda sexta no fim do dia",
    ],
    "canal": ["no Teams", "por e-mail", "aqui no chat", "por notificação"],
    "dias": ["5", "7", "10", "15", "30", "45", "60"],
    "pct": ["5%", "10%", "15%", "20%", "25%", "80%", "90%"],
    "status": [
        "em andamento", "concluído", "paralisado", "em homologação",
        "aguardando aprovação", "cancelado", "em planejamento",
    ],
    "data": [
        "30/06/2026", "15/09/2026", "dezembro de 2026", "o fim do trimestre",
        "31/03/2027", "a próxima quinzena",
    ],
    "risco": [
        "atraso na desapropriação", "indisponibilidade de fornecedor",
        "estouro de orçamento", "risco ambiental", "risco de integração de sistemas",
        "falta de mão de obra especializada", "risco regulatório",
    ],
    "tema_ext": [
        "a previsão do tempo para amanhã", "o resultado do jogo de ontem",
        "uma receita de bolo", "quantos dias de férias eu tenho",
        "a cotação do dólar", "uma piada", "como declarar imposto de renda",
        "o horário do último trem para casa", "quem ganhou o campeonato",
    ],
}

TOTAL_COMB = {k: len(v) for k, v in SLOTS.items()}
print("Slots disponíveis:", TOTAL_COMB)

Slots disponíveis: {'proj': 24, 'sub': 8, 'per': 9, 'resp': 10, 'doc': 9, 'crit': 8, 'ind': 7, 'freq': 7, 'canal': 4, 'dias': 7, 'pct': 7, 'status': 7, 'data': 6, 'risco': 7, 'tema_ext': 9}


In [4]:
# ============================================================
# 2.2 Templates de prompts por intenção específica
#     (o dicionário abaixo é o "dicionário extenso de prompts")
# ============================================================

TEMPLATES = {
# ------------------------------- CONSULTA -------------------------------
"consulta_status_projeto": [
    "qual é o status do {proj}?",
    "como está o andamento do {proj}?",
    "me diga a situação atual do {proj}",
    "o {proj} está em que fase agora?",
    "queria saber como anda o {proj} {per}",
    "o {proj} já foi concluído?",
    "situação do {proj}, por favor",
    "me atualiza sobre o {proj}",
    "qual o percentual de avanço do {proj}?",
    "o {proj} continua {status}?",
    "quais projetos do {sub} estão {status}?",
    "tem alguma novidade sobre o {proj}?",
    "resumo rápido da situação do {proj}",
    "o {proj} saiu do papel?",
    "em que pé está o {proj}?",
],
"consulta_cronograma_prazo": [
    "qual é o prazo do {proj}?",
    "quando o {proj} termina?",
    "qual a data prevista de conclusão do {proj}?",
    "o {proj} está atrasado?",
    "quantos dias de atraso o {proj} acumula?",
    "a entrega do {proj} ainda é para {data}?",
    "me mostra o cronograma do {proj}",
    "qual a data de início do {proj}?",
    "o {proj} vai cumprir o prazo de {data}?",
    "quais projetos do {sub} estão com prazo estourado?",
    "quanto tempo falta para o {proj} acabar?",
    "houve replanejamento de datas no {proj}?",
    "a linha de base do cronograma do {proj} mudou?",
    "qual a duração total prevista para o {proj}?",
    "o {proj} tem previsão de término {per}?",
],
"consulta_marcos_entregas": [
    "quais são os marcos do {proj}?",
    "quais entregas do {proj} estão previstas {per}?",
    "o marco de {data} do {proj} foi cumprido?",
    "me lista as próximas entregas do {proj}",
    "quantos marcos do {proj} já foram concluídos?",
    "quais milestones do {sub} vencem {per}?",
    "qual o próximo marco do {proj}?",
    "as entregas do {proj} estão em dia?",
    "quais etapas do {proj} ainda faltam?",
    "houve marco não cumprido no {proj}?",
    "cronograma de entregas do {proj}, por favor",
    "quais são as entregas contratuais do {proj}?",
    "quantas entregas do {sub} foram concluídas {per}?",
    "quando é a próxima entrega do {proj}?",
],
"consulta_orcamento_custo": [
    "qual é o orçamento do {proj}?",
    "quanto já foi gasto no {proj}?",
    "qual o custo realizado do {proj} {per}?",
    "o {proj} está dentro do orçamento?",
    "qual o saldo orçamentário do {proj}?",
    "quanto o {proj} estourou do previsto?",
    "qual o desvio financeiro do {proj}?",
    "quanto custa o {sub} {per}?",
    "qual o valor aprovado no TAP do {proj}?",
    "me mostra a execução orçamentária do {proj}",
    "quanto ainda falta desembolsar no {proj}?",
    "o custo do {proj} subiu {per}?",
    "qual o comprometimento financeiro do {sub}?",
    "quanto do orçamento do {proj} já foi consumido?",
],
"consulta_escopo_tap": [
    "qual é o escopo do {proj}?",
    "o que está previsto no TAP do {proj}?",
    "quais são os objetivos do {proj}?",
    "quais premissas foram registradas no {proj}?",
    "quais restrições constam no TAP do {proj}?",
    "o que o {proj} entrega no final?",
    "me explica o escopo do {proj} em poucas linhas",
    "quais são os benefícios esperados do {proj}?",
    "o {proj} inclui obra civil no escopo?",
    "houve mudança de escopo no {proj}?",
    "qual a justificativa de abertura do {proj}?",
    "quais são os critérios de sucesso do {proj}?",
    "o que está fora do escopo do {proj}?",
    "qual o objetivo estratégico associado ao {proj}?",
],
"consulta_responsavel_equipe": [
    "quem é o responsável pelo {proj}?",
    "quem gerencia o {proj}?",
    "qual área conduz o {proj}?",
    "quem é o patrocinador do {proj}?",
    "com quem eu falo sobre o {proj}?",
    "quais projetos estão sob responsabilidade de {resp}?",
    "quem responde pelo {sub}?",
    "qual o time alocado no {proj}?",
    "quem aprovou o TAP do {proj}?",
    "quantas pessoas estão no {proj}?",
    "quem é o ponto focal do {proj} na GTI?",
    "{resp} é responsável por qual projeto mesmo?",
    "qual gerente cuida do {proj}?",
    "quem assinou o termo de abertura do {proj}?",
],
"consulta_indicadores_kpi": [
    "qual o {ind} do {proj}?",
    "me mostra os indicadores do {proj}",
    "como está o {ind} do {sub} {per}?",
    "qual o percentual de conclusão do {proj}?",
    "o {ind} do {proj} está dentro da meta?",
    "quais KPIs o {proj} acompanha?",
    "qual a aderência ao cronograma do {sub}?",
    "me dá o painel de indicadores do {proj}",
    "qual o índice de desempenho de custo do {proj}?",
    "os indicadores do {sub} melhoraram {per}?",
    "qual a curva S do {proj}?",
    "qual a meta de {ind} do {proj} para {per}?",
    "quantos projetos do {sub} estão com {ind} abaixo da meta?",
    "qual o desempenho geral do {proj} em números?",
],
"consulta_riscos_projeto": [
    "quais são os riscos do {proj}?",
    "o {proj} tem algum risco crítico?",
    "me mostra a matriz de riscos do {proj}",
    "qual o risco de maior impacto no {proj}?",
    "o risco de {risco} está mapeado no {proj}?",
    "quantos riscos abertos existem no {sub}?",
    "qual o plano de tratamento do risco de {risco} no {proj}?",
    "quais riscos do {proj} foram materializados {per}?",
    "existe risco de {risco} em algum projeto do {sub}?",
    "qual a probabilidade do risco de {risco} no {proj}?",
    "os riscos do {proj} foram revisados {per}?",
    "quais projetos têm risco alto de {risco}?",
    "me lista os riscos mapeados do {sub}",
    "qual a exposição a risco do {proj}?",
],
"consulta_pendencias_projeto": [
    "quais pendências estão abertas no {proj}?",
    "o que está travando o {proj}?",
    "tem alguma ação em aberto no {proj}?",
    "quantas pendências o {sub} acumula?",
    "quais itens do {proj} aguardam resposta de {resp}?",
    "quais pendências do {proj} venceram {per}?",
    "o que falta resolver no {proj}?",
    "quais ações estão atrasadas no {sub}?",
    "existe algo pendente de aprovação no {proj}?",
    "me lista os itens em aberto do {proj}",
    "quais pendências estão com {resp}?",
    "há bloqueios registrados no {proj}?",
    "quais decisões estão pendentes no {proj}?",
    "quantas ações corretivas seguem abertas no {proj}?",
],
"consulta_documentos_fonte": [
    "onde está o {doc} do {proj}?",
    "me manda o link do {doc} do {proj}",
    "qual a fonte da informação de custo do {proj}?",
    "de onde vem o dado de prazo do {proj}?",
    "existe {doc} disponível para o {proj}?",
    "qual a versão mais recente do {doc} do {proj}?",
    "em qual pasta do SharePoint está o {doc} do {sub}?",
    "quem enviou o {doc} do {proj}?",
    "quando o {doc} do {proj} foi atualizado pela última vez?",
    "quais documentos existem para o {proj}?",
    "me mostra a referência usada para responder sobre o {proj}",
    "o {doc} do {proj} está anexado onde?",
    "consigo acessar o {doc} do {sub}?",
    "qual documento comprova o avanço do {proj}?",
],
"consulta_visao_portfolio": [
    "quantos projetos existem no portfólio?",
    "me dá uma visão geral do {sub}",
    "quantos projetos estão {status} no portfólio?",
    "qual o total investido no portfólio {per}?",
    "quais são os projetos do {sub}?",
    "como está o portfólio {per}?",
    "quantos programas o Metrô conduz atualmente?",
    "me mostra o panorama do portfólio {per}",
    "qual subportfólio tem mais projetos?",
    "resumo executivo do portfólio, por favor",
    "quantos projetos do {sub} estão atrasados?",
    "qual a distribuição dos projetos por status?",
    "quantos projetos entraram no portfólio {per}?",
    "qual o volume financeiro total do {sub}?",
],
"comparacao_entre_projetos": [
    "compare o {proj} com o {proj2}",
    "qual está mais avançado, o {proj} ou o {proj2}?",
    "diferença de {crit} entre o {proj} e o {proj2}",
    "me mostra {proj} versus {proj2} em {crit}",
    "entre {proj} e {proj2}, qual tem maior risco?",
    "compare os projetos do {sub} por {crit}",
    "qual projeto gastou mais, o {proj} ou o {proj2}?",
    "faça um comparativo de {crit} entre {proj} e {proj2}",
    "o {proj} está melhor que o {proj2} em {crit}?",
    "quais semelhanças existem entre o {proj} e o {proj2}?",
    "ranking dos projetos do {sub} por {crit}",
    "compare o desempenho do {sub} com o do {proj}",
    "coloque {proj} e {proj2} lado a lado",
    "qual dos dois entrega antes, {proj} ou {proj2}?",
],
"analise_tendencia_desempenho": [
    "o desempenho do {proj} melhorou {per}?",
    "qual a tendência de custo do {sub} {per}?",
    "o atraso do {proj} está aumentando?",
    "analise a evolução do {ind} do {proj}",
    "por que o {proj} piorou {per}?",
    "qual a projeção de término do {proj} mantido o ritmo atual?",
    "o {sub} tende a estourar o orçamento?",
    "faça uma análise do comportamento do {proj} {per}",
    "quais fatores explicam o desvio do {proj}?",
    "o {ind} do {sub} está em queda?",
    "qual a perspectiva do {proj} para {per}?",
    "analise o histórico de replanejamentos do {proj}",
    "existe padrão de atraso no {sub}?",
    "qual a projeção de gasto do {proj} até {data}?",
],
# ------------------------------- TRANSACAO -------------------------------
"transacao_atualizar_status": [
    "atualize o status do {proj} para {status}",
    "marque o {proj} como {status}",
    "mude a situação do {proj} para {status}",
    "registre que o {proj} está {status}",
    "altere o percentual de avanço do {proj} para {pct}",
    "atualiza aí: {proj} agora está {status}",
    "quero atualizar o andamento do {proj}",
    "coloque o {proj} como {status} no sistema",
    "por favor, registre o avanço de {pct} no {proj}",
    "faça a atualização de status do {sub}",
    "grave o novo status do {proj}",
    "reabra o {proj}, ele voltou a ficar {status}",
    "informe no portfólio que o {proj} está {status}",
    "atualizar situação do {proj}",
],
"transacao_atualizar_cronograma": [
    "prorrogue o prazo do {proj} para {data}",
    "altere a data de término do {proj} para {data}",
    "replaneje o cronograma do {proj}",
    "adie o marco do {proj} em {dias} dias",
    "registre o replanejamento do {proj}",
    "mude a data de entrega do {proj}",
    "antecipe a entrega do {proj} para {data}",
    "quero repactuar as datas do {proj}",
    "atualize a linha de base do cronograma do {proj}",
    "empurre as entregas do {proj} em {dias} dias",
    "reprograme os marcos do {sub}",
    "salve a nova data de conclusão do {proj}",
    "ajuste o cronograma do {proj} conforme o replanejamento",
    "cadastre o novo marco do {proj} para {data}",
],
"transacao_registrar_risco": [
    "registre o risco de {risco} no {proj}",
    "cadastre um novo risco no {proj}",
    "inclua {risco} na matriz de riscos do {proj}",
    "atualize a probabilidade do risco de {risco} no {proj}",
    "eleve a criticidade do risco de {risco} no {proj}",
    "quero registrar um risco no {sub}",
    "adicione o plano de tratamento do risco de {risco} no {proj}",
    "encerre o risco de {risco} do {proj}",
    "documente o risco de {risco} identificado {per}",
    "abra um risco novo para o {proj}",
    "classifique o risco de {risco} do {proj} como alto",
    "registre a materialização do risco de {risco} no {proj}",
    "grave na matriz de riscos do {proj} o item de {risco}",
    "atualize o impacto do risco de {risco}",
],
"transacao_registrar_pendencia": [
    "abra uma pendência no {proj}",
    "atribua a pendência do {proj} para {resp}",
    "registre uma ação para {resp} no {proj}",
    "encerre a pendência do {proj}",
    "crie um item de ação no {proj} com prazo de {dias} dias",
    "marque a pendência do {proj} como resolvida",
    "quero registrar um bloqueio no {proj}",
    "reatribua as pendências do {sub} para {resp}",
    "adicione uma tarefa de acompanhamento no {proj}",
    "documente a decisão tomada na reunião do {proj}",
    "registre o item pendente de aprovação do {proj}",
    "atualize o prazo da pendência do {proj} para {data}",
    "cadastre a ação corretiva do {proj}",
    "feche todas as pendências concluídas do {proj}",
],
"transacao_gerar_relatorio": [
    "gere o relatório anual do {sub}",
    "monte um relatório de acompanhamento do {proj}",
    "quero um relatório consolidado do portfólio {per}",
    "gera aí o balanço do {sub}",
    "prepare o relatório de fechamento do {proj}",
    "emita o relatório de status do {proj} {per}",
    "produza um resumo escrito do {sub} para a diretoria",
    "faça o relatório de riscos do {proj}",
    "gere o relatório financeiro do {proj} {per}",
    "monte o documento de prestação de contas do {sub}",
    "quero o relatório de partida do portfólio",
    "elabore um relatório executivo do {proj}",
    "gere o relatório de pendências do {sub}",
    "crie o relatório mensal do {proj}",
],
"transacao_gerar_apresentacao": [
    "gere uma apresentação do {sub}",
    "monte os slides do {proj} para a reunião",
    "crie um deck executivo sobre o portfólio {per}",
    "quero uma apresentação do {proj} no template padrão",
    "prepare slides de status do {sub}",
    "gera uma apresentação de 5 slides sobre o {proj}",
    "monte o material da reunião de portfólio {per}",
    "crie a apresentação de fechamento do {proj}",
    "faça um deck comparando os projetos do {sub}",
    "quero slides com os indicadores do {proj}",
    "prepare a apresentação para a diretoria sobre o {proj}",
    "gere o PPT de acompanhamento do {sub}",
    "monte a apresentação usando o template do PMO",
    "crie slides com os riscos do {proj}",
],
"transacao_exportar_dados": [
    "exporte os dados do {sub} em planilha",
    "baixe a lista de projetos do portfólio em excel",
    "quero um csv com os marcos do {proj}",
    "exporta os indicadores do {sub} para xlsx",
    "gere um arquivo com o cronograma do {proj}",
    "me manda os dados do {proj} em formato de tabela",
    "exportar riscos do {sub}",
    "quero baixar o histórico financeiro do {proj}",
    "salve os dados do portfólio em uma planilha",
    "extraia os dados de custo do {sub} {per}",
    "gere um dump dos projetos {status}",
    "exporte as pendências do {proj} para excel",
    "quero um arquivo com todos os projetos do {sub}",
    "gera a planilha de acompanhamento do {proj}",
],
"transacao_agendar_compromisso": [
    "agende uma reunião sobre o {proj}",
    "marque um checkpoint do {proj} para {data}",
    "agenda a revisão do portfólio {per}",
    "crie um convite para a reunião do {sub}",
    "marque uma call com {resp} sobre o {proj}",
    "agende o comitê de portfólio para {data}",
    "reserve uma hora para tratar do {proj}",
    "coloque na agenda a revisão de riscos do {proj}",
    "marque a reunião de fechamento do {proj}",
    "agende {freq} um ponto de controle do {sub}",
    "crie um compromisso com a equipe do {proj}",
    "remarque a reunião do {proj} para {data}",
    "cancele a reunião do {proj}",
    "agenda com {resp} uma conversa sobre o {sub}",
],
"transacao_solicitar_atualizacao": [
    "peça a {resp} a atualização do {proj}",
    "envie uma cobrança de atualização do {proj} {canal}",
    "solicite ao responsável do {proj} os dados de {per}",
    "manda um lembrete para {resp} atualizar o cronograma do {proj}",
    "cobre o {doc} pendente do {proj}",
    "notifique {resp} que o {proj} está sem atualização",
    "peça os dados de custo do {sub} para {resp}",
    "envie a solicitação de atualização do {sub} {canal}",
    "dispare um pedido de atualização para os gerentes do {sub}",
    "solicite a revisão da matriz de riscos do {proj}",
    "mande mensagem para {resp} sobre as pendências do {proj}",
    "peça confirmação do marco de {data} ao responsável do {proj}",
    "cobre {resp} sobre o relatório do {proj}",
    "solicite o preenchimento do {doc} do {proj}",
],
"transacao_cadastrar_projeto": [
    "cadastre um novo projeto no {sub}",
    "quero incluir um projeto no portfólio",
    "abra um novo programa no {sub}",
    "registre o {proj} como novo projeto do portfólio",
    "crie o cadastro inicial do {proj}",
    "inclua um novo item no {sub}",
    "quero abrir o TAP de um projeto novo",
    "adicione um projeto ao portfólio com início em {data}",
    "cadastrar novo empreendimento no {sub}",
    "registre a entrada do {proj} no portfólio",
    "crie a ficha do {proj} no sistema",
    "quero cadastrar um programa novo com {resp} como responsável",
    "inicie o cadastro de um projeto no {sub}",
    "adicione o {proj} à carteira",
],
# ------------------------------- ALERTA -------------------------------
"alerta_prazo_atraso": [
    "me avise se o {proj} atrasar",
    "quero ser notificado quando algum projeto do {sub} passar do prazo",
    "avise quando o {proj} ficar com mais de {dias} dias de atraso",
    "me alerta se o marco de {data} do {proj} não for cumprido",
    "notifique atraso em qualquer projeto do portfólio",
    "quero alerta de prazo estourado no {sub}",
    "me avisa se a entrega do {proj} escorregar",
    "avise {canal} quando o cronograma do {proj} atrasar",
    "manda alerta se o {proj} não terminar até {data}",
    "quero saber assim que algum marco do {sub} vencer",
    "me notifique sobre atrasos {freq}",
    "alerta de prazo para o {proj}, por favor",
    "avise se o atraso do {proj} passar de {dias} dias",
    "quero ser avisado de qualquer descumprimento de prazo no {sub}",
],
"alerta_estouro_orcamento": [
    "me avise se o {proj} estourar o orçamento",
    "quero alerta quando o custo do {proj} passar de {pct} do previsto",
    "notifique desvio orçamentário no {sub}",
    "avise se algum projeto do {sub} ultrapassar o orçamento aprovado",
    "quero ser alertado sobre gastos acima do planejado no {proj}",
    "manda alerta de execução orçamentária do {sub} {freq}",
    "me avisa se o saldo do {proj} ficar abaixo de {pct}",
    "notifique {canal} quando houver estouro de custo",
    "quero alerta financeiro do {proj}",
    "avise se o CPI do {proj} cair abaixo da meta",
    "me alerte sobre qualquer desvio de custo acima de {pct}",
    "quero ser notificado se o {sub} exceder o orçamento {per}",
    "configure aviso de estouro orçamentário para o portfólio",
    "avise se o custo do {proj} crescer {pct} {per}",
],
"alerta_risco_critico": [
    "me avise se surgir risco crítico no {proj}",
    "quero ser notificado quando um risco do {sub} for escalado",
    "alerte quando o risco de {risco} se materializar",
    "notifique riscos de alto impacto no portfólio",
    "avise se a criticidade de algum risco do {proj} aumentar",
    "quero alerta de novos riscos no {sub}",
    "me avisa se aparecer risco vermelho em qualquer projeto",
    "manda alerta quando o risco de {risco} passar para alto",
    "notifique {canal} sobre riscos críticos {freq}",
    "quero ser alertado se o {proj} entrar em zona de risco",
    "avise sobre riscos não tratados no {sub}",
    "alerta de risco para o {proj}, por favor",
    "me notifique se algum risco do portfólio virar problema",
    "quero saber na hora se houver risco crítico no {sub}",
],
"alerta_desatualizacao_dados": [
    "me avise se o {proj} ficar {dias} dias sem atualização",
    "quero alerta quando o {doc} do {proj} estiver desatualizado",
    "notifique projetos do {sub} sem atualização {per}",
    "avise se algum gerente não atualizar os dados {freq}",
    "quero ser alertado sobre informações vencidas no portfólio",
    "manda aviso quando o cronograma do {proj} ficar defasado",
    "me avisa se o {sub} tiver dados faltando",
    "alerta de base desatualizada, por favor",
    "notifique quando o status do {proj} não for revisado {per}",
    "quero saber quais projetos estão com informação velha",
    "avise se o {doc} do {sub} não for enviado até {data}",
    "configure aviso de pendência documental no {proj}",
    "me alerte sobre campos obrigatórios em branco no {sub}",
    "quero alerta de projetos parados sem justificativa",
],
"alerta_configurar_monitoramento": [
    "configure um alerta {freq} para o {sub}",
    "crie uma regra de monitoramento do {proj}",
    "quero receber um resumo do portfólio {freq} {canal}",
    "programe um aviso {freq} sobre o {proj}",
    "configure notificação de {ind} abaixo da meta",
    "crie um monitoramento automático do {sub}",
    "quero um digest {freq} dos projetos {status}",
    "configure o envio de alertas {canal}",
    "monte uma regra: se o {ind} do {proj} cair, me avise",
    "programe o acompanhamento automático do {proj}",
    "quero configurar gatilhos de alerta para o portfólio",
    "defina {freq} como frequência dos meus avisos",
    "crie um alerta personalizado para o {sub}",
    "configure notificação {canal} para o {proj}",
],
"alerta_gerenciar_inscricao": [
    "cancele os alertas do {proj}",
    "pare de me notificar sobre o {sub}",
    "silencie os avisos do {proj} por {dias} dias",
    "quero desativar todos os meus alertas",
    "reative os alertas do {sub}",
    "quais alertas eu tenho configurados?",
    "remova a notificação de atraso do {proj}",
    "pause os avisos {canal}",
    "desinscreva {resp} dos alertas do {proj}",
    "mude a frequência dos meus alertas para {freq}",
    "não quero mais receber aviso de risco do {sub}",
    "liste minhas regras de monitoramento ativas",
    "inscreva {resp} nos alertas do {proj}",
    "desligue os alertas de orçamento do {proj}",
],
# ---------------------------- FORA DE ESCOPO ----------------------------
"fora_escopo_social": [
    "bom dia",
    "boa tarde, tudo bem?",
    "olá",
    "oi, você está aí?",
    "obrigado pela ajuda",
    "valeu!",
    "quem é você?",
    "o que você consegue fazer?",
    "testando",
    "tchau, até logo",
    "você é um robô?",
    "legal, gostei da resposta",
    "me ajuda?",
    "boa noite",
    "só testando o agente aqui",
],
"fora_escopo_externo": [
    "qual {tema_ext}?",
    "me diz {tema_ext}",
    "você sabe {tema_ext}?",
    "preciso saber {tema_ext}",
    "consulta pra mim {tema_ext}",
    "quero saber {tema_ext}",
    "me ajuda com {tema_ext}",
    "pesquisa {tema_ext}",
    "qual a sua opinião sobre política?",
    "me conta uma curiosidade qualquer",
    "traduz essa frase para o inglês",
    "resolve essa conta de matemática pra mim",
    "escreve um poema",
    "qual o melhor restaurante perto daqui?",
],
}

assert set(TEMPLATES) == set(INTENCOES), (
    "Intenções sem template: " + str(set(INTENCOES) - set(TEMPLATES)) +
    " | templates órfãos: " + str(set(TEMPLATES) - set(INTENCOES))
)

n_templates = sum(len(v) for v in TEMPLATES.values())
print(f"{n_templates} templates para {len(TEMPLATES)} intenções "
      f"(mín {min(len(v) for v in TEMPLATES.values())}, máx {max(len(v) for v in TEMPLATES.values())} por intenção)")

437 templates para 31 intenções (mín 14, máx 15 por intenção)


In [5]:
# ============================================================
# 2.3 Expansão dos templates -> dataset de prompts
# ============================================================
from string import Formatter

N_VARIANTES = 14          # variantes geradas por template
P_VARIACAO_SUPERFICIAL = 0.45   # fração de variantes que recebem ruído de digitação/estilo


def _campos(tpl):
    return [f for _, f, _, _ in Formatter().parse(tpl) if f]


def _sem_acento(txt):
    return "".join(c for c in unicodedata.normalize("NFD", txt)
                   if unicodedata.category(c) != "Mn")


def variacao_superficial(txt, rng):
    """Ruído de estilo aplicado DENTRO do grupo (template) — nunca cria grupo novo."""
    op = rng.choice(["sem_acento", "sem_interrogacao", "capitalizado", "cortesia",
                     "informal", "maiusculas"])
    if op == "sem_acento":
        return _sem_acento(txt)
    if op == "sem_interrogacao":
        return txt.rstrip("?").strip()
    if op == "capitalizado":
        return txt[:1].upper() + txt[1:]
    if op == "cortesia":
        return rng.choice(["por favor, ", "pfv ", "preciso que vc me diga: ", "poderia me informar: "]) + txt
    if op == "informal":
        t = txt.replace("você", "vc").replace("por favor", "pfv").replace("para", "pra")
        return t
    if op == "maiusculas":
        return txt.upper()
    return txt


def gerar_dataset(templates, slots, n_variantes=N_VARIANTES, seed=SEED):
    linhas = []
    for intencao, tpls in templates.items():
        macro = INTENCAO2MACRO[intencao]
        for t_idx, tpl in enumerate(tpls):
            template_id = f"{intencao}#{t_idx:02d}"
            rng = random.Random(f"{seed}-{template_id}")
            campos = _campos(tpl)
            vistos = set()
            tentativas = 0
            while len(vistos) < n_variantes and tentativas < n_variantes * 40:
                tentativas += 1
                valores = {}
                for c in campos:
                    if c == "proj2":
                        valores[c] = rng.choice(slots["proj"])
                    else:
                        valores[c] = rng.choice(slots[c])
                if "proj" in valores and "proj2" in valores and valores["proj"] == valores["proj2"]:
                    continue
                texto = tpl.format(**valores)
                texto = re.sub(r"\s+", " ", texto).strip()
                variacao = "base"
                if rng.random() < P_VARIACAO_SUPERFICIAL:
                    texto_v = variacao_superficial(texto, rng)
                    if texto_v != texto:
                        texto, variacao = texto_v, "ruido"
                if texto.lower() in vistos:
                    continue
                vistos.add(texto.lower())
                linhas.append({
                    "texto": texto,
                    "intencao": intencao,
                    "macrointencao": macro,
                    "template_id": template_id,
                    "variacao": variacao,
                })
    return pd.DataFrame(linhas)


df = gerar_dataset(TEMPLATES, SLOTS)


def normalizar(txt):
    """Chave de deduplicação: minúsculas, sem acento, sem pontuação, espaços colapsados."""
    t = _sem_acento(txt.lower())
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


df["chave"] = df["texto"].map(normalizar)
antes = len(df)
df = df.drop_duplicates(subset="chave").reset_index(drop=True)

print(f"{antes} prompts gerados -> {len(df)} após deduplicação exata ({antes - len(df)} removidos)")
print(f"{df['template_id'].nunique()} templates | {df['intencao'].nunique()} intenções | {df['macrointencao'].nunique()} macrointenções")
display(df.sample(12, random_state=SEED)[["texto", "intencao", "macrointencao", "template_id"]])

5763 prompts gerados -> 5353 após deduplicação exata (410 removidos)
437 templates | 31 intenções | 4 macrointenções


,texto,intencao,macrointencao,template_id
5010,pare de me notificar sobre o carteira de obras...,alerta_gerenciar_inscricao,ALERTA,alerta_gerenciar_inscricao#01
1650,quantas ações corretivas seguem abertas no Lin...,consulta_pendencias_projeto,CONSULTA,consulta_pendencias_projeto#13
401,quais são os marcos do modernização do CCO?,consulta_marcos_entregas,CONSULTA,consulta_marcos_entregas#00
4222,pfv quero alerta de prazo estourado no subport...,alerta_prazo_atraso,ALERTA,alerta_prazo_atraso#05
1684,me manda o link do relatório de acompanhamento...,consulta_documentos_fonte,CONSULTA,consulta_documentos_fonte#01
33,me diga a situação atual do modernização da Li...,consulta_status_projeto,CONSULTA,consulta_status_projeto#02
290,me mostra o cronograma do PRJ-1187,consulta_cronograma_prazo,CONSULTA,consulta_cronograma_prazo#06
3265,"por favor, gere o relatório de pendências do s...",transacao_gerar_relatorio,TRANSACAO,transacao_gerar_relatorio#12
5325,poderia me informar: qual a sua opinião sobre ...,fora_escopo_externo,FORA_DE_ESCOPO,fora_escopo_externo#08
3235,poderia me informar: monte o documento de pres...,transacao_gerar_relatorio,TRANSACAO,transacao_gerar_relatorio#09


In [6]:
# ============================================================
# 2.4 Distribuição do dataset
# ============================================================
dist_macro = df["macrointencao"].value_counts()
dist_int = df["intencao"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
dist_macro.plot(kind="bar", ax=axes[0], color="#2b6cb0")
axes[0].set_title("Prompts por macrointenção")
axes[0].set_ylabel("nº de prompts")
axes[0].tick_params(axis="x", rotation=20)

dist_int.sort_values().plot(kind="barh", ax=axes[1], color="#38a169")
axes[1].set_title("Prompts por intenção específica")
axes[1].tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

print("Comprimento médio (palavras):", round(df["texto"].str.split().str.len().mean(), 1))
print("\nDesbalanceamento macro (razão maior/menor classe):",
      round(dist_macro.max() / dist_macro.min(), 2))
print("Desbalanceamento intenção (razão maior/menor classe):",
      round(dist_int.max() / dist_int.min(), 2))

Comprimento médio (palavras): 10.2

Desbalanceamento macro (razão maior/menor classe): 11.43
Desbalanceamento intenção (razão maior/menor classe): 2.53


## 3. Divisão treino / validação / teste **sem vazamento**

Em datasets de intenção gerados por template, o vazamento acontece de três formas. Todas são tratadas aqui:

| Fonte de vazamento | Tratamento |
|---|---|
| Mesma frase (ou variação apenas de acento/caixa) em dois splits | Deduplicação pela chave normalizada, feita **antes** do split |
| Mesmo template gerando frases quase idênticas em treino e teste | Split **por grupo** (`template_id`): um template pertence a um único split |
| Ajuste de hiperparâmetro/pré-processamento olhando o teste | Toda seleção usa **treino (CV agrupada)** e **validação**; o teste é aberto uma única vez, no final |

Também deixamos calculado um **split aleatório ingênuo** (por linha), usado apenas para *demonstrar* o quanto ele infla a métrica — ele nunca é usado para escolher nada.

In [7]:
# ============================================================
# 3.1 Split agrupado por template (60 / 20 / 20 dos templates de cada intenção)
# ============================================================
PROP = {"treino": 0.60, "validacao": 0.20, "teste": 0.20}

def split_por_template(df, seed=SEED):
    atribuicao = {}
    rng = random.Random(seed)
    for intencao, grupo in df.groupby("intencao"):
        tpls = sorted(grupo["template_id"].unique())
        rng.shuffle(tpls)
        n = len(tpls)
        n_val = max(1, int(round(n * PROP["validacao"])))
        n_test = max(1, int(round(n * PROP["teste"])))
        n_train = n - n_val - n_test
        assert n_train >= 1, f"Intenção {intencao} tem templates demais de menos"
        for t in tpls[:n_train]:
            atribuicao[t] = "treino"
        for t in tpls[n_train:n_train + n_val]:
            atribuicao[t] = "validacao"
        for t in tpls[n_train + n_val:]:
            atribuicao[t] = "teste"
    return atribuicao

atribuicao = split_por_template(df)
df["split"] = df["template_id"].map(atribuicao)

# Split aleatório ingênuo — SÓ para demonstração do vazamento (nunca usado para decidir nada)
idx = np.random.RandomState(SEED).permutation(len(df))
corte1, corte2 = int(0.6 * len(df)), int(0.8 * len(df))
df["split_ingenuo"] = "teste"
df.loc[df.index[idx[:corte1]], "split_ingenuo"] = "treino"
df.loc[df.index[idx[corte1:corte2]], "split_ingenuo"] = "validacao"

tab = pd.crosstab(df["macrointencao"], df["split"])
tab["total"] = tab.sum(axis=1)
display(tab)

print(df["split"].value_counts().to_string())
print("\nIntenções presentes em cada split:",
      {s: df[df.split == s]["intencao"].nunique() for s in ["treino", "validacao", "teste"]},
      f"(de {len(INTENCOES)})")

split,teste,treino,validacao,total
macrointencao,,,,
ALERTA,202,562,229,993
CONSULTA,492,1353,510,2355
FORA_DE_ESCOPO,46,124,36,206
TRANSACAO,395,1023,381,1799


split
treino       3062
validacao    1156
teste        1135

Intenções presentes em cada split: {'treino': 31, 'validacao': 31, 'teste': 31} (de 31)


In [8]:
# ============================================================
# 3.2 Auditoria de vazamento (executada ANTES de qualquer treino)
# ============================================================
from sklearn.metrics.pairwise import cosine_similarity

treino = df[df.split == "treino"].reset_index(drop=True)
validacao = df[df.split == "validacao"].reset_index(drop=True)
teste = df[df.split == "teste"].reset_index(drop=True)

falhas = []

# (1) Nenhum template compartilhado entre splits
for a, b in [("treino", "validacao"), ("treino", "teste"), ("validacao", "teste")]:
    inter = set(df[df.split == a]["template_id"]) & set(df[df.split == b]["template_id"])
    if inter:
        falhas.append(f"templates compartilhados entre {a} e {b}: {sorted(inter)[:5]}")

# (2) Nenhum texto normalizado repetido entre splits
for a, b in [("treino", "validacao"), ("treino", "teste"), ("validacao", "teste")]:
    inter = set(df[df.split == a]["chave"]) & set(df[df.split == b]["chave"])
    if inter:
        falhas.append(f"{len(inter)} textos idênticos entre {a} e {b}")

# (3) Todas as intenções representadas em todos os splits
for s in ["treino", "validacao", "teste"]:
    faltando = set(INTENCOES) - set(df[df.split == s]["intencao"])
    if faltando:
        falhas.append(f"intenções ausentes em {s}: {sorted(faltando)}")

# (4) Quase-duplicatas entre treino e teste (similaridade de cosseno em TF-IDF de caracteres)
vec_check = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
X_all = vec_check.fit_transform(pd.concat([treino["chave"], teste["chave"]]))
X_tr, X_te = X_all[:len(treino)], X_all[len(treino):]
sim_max = np.asarray(cosine_similarity(X_te, X_tr).max(axis=1)).ravel()
LIMIAR_QUASE_DUP = 0.97
n_quase_dup = int((sim_max >= LIMIAR_QUASE_DUP).sum())

print("=" * 68)
print("AUDITORIA DE VAZAMENTO")
print("=" * 68)
print(f"templates: treino={treino.template_id.nunique()} | val={validacao.template_id.nunique()} | teste={teste.template_id.nunique()}")
print(f"exemplos : treino={len(treino)} | val={len(validacao)} | teste={len(teste)}")
print(f"similaridade máxima teste->treino: média={sim_max.mean():.3f} | máx={sim_max.max():.3f}")
print(f"quase-duplicatas (sim >= {LIMIAR_QUASE_DUP}): {n_quase_dup} ({100*n_quase_dup/len(teste):.2f}% do teste)")

if n_quase_dup:
    manter = sim_max < LIMIAR_QUASE_DUP
    teste = teste[manter].reset_index(drop=True)
    print(f"-> {n_quase_dup} exemplos removidos do teste por precaução; teste final = {len(teste)}")

print("\nRESULTADO:", "OK — nenhum vazamento detectado" if not falhas else "FALHAS: " + " | ".join(falhas))
assert not falhas, falhas

plt.figure(figsize=(7, 4))
plt.hist(sim_max, bins=40, color="#805ad5")
plt.axvline(LIMIAR_QUASE_DUP, color="red", ls="--", label=f"limiar {LIMIAR_QUASE_DUP}")
plt.title("Similaridade máxima de cada exemplo de teste com o treino")
plt.xlabel("cosseno (TF-IDF de caracteres)"); plt.ylabel("frequência"); plt.legend()
plt.tight_layout(); plt.show()

AUDITORIA DE VAZAMENTO
templates: treino=251 | val=93 | teste=93
exemplos : treino=3062 | val=1156 | teste=1135
similaridade máxima teste->treino: média=0.584 | máx=0.906
quase-duplicatas (sim >= 0.97): 0 (0.00% do teste)

RESULTADO: OK — nenhum vazamento detectado


## 4. Pré-processamento 

A função `preprocess_text` e a grade das **36 configurações** vêm do notebook de pré-processamento do grupo (tokenizador, remoção de stopwords, stemming Snowball/RSLP, n-gramas).

A escolha da melhor configuração é feita com `StratifiedGroupKFold` **apenas sobre o conjunto de treino**, agrupando por `template_id` — ou seja, mesmo dentro da validação cruzada nenhum fold vê um template que aparece em outro. Validação e teste não participam dessa escolha.

Na execução completa de referência, venceu a **`config_22`**: texto em minúsculas, tokenização por expressão regular (`[^\W_]+`), preservação de stopwords, remoção de pontuação e stemming Snowball em português, sem geração adicional de n-gramas. A preservação de stopwords é coerente com o domínio, pois termos como “qual”, “quando”, “quanto” e “quem” ajudam a distinguir intenções de consulta.

In [9]:
# ============================================================
# 4.1 preprocess_text — idêntica à do notebook de pré-processamento do grupo
# ============================================================
def preprocess_text(
    text,
    lowercase=True,
    tokenizer="word",
    remove_stopwords=False,
    remove_punctuation=True,
    morphology="none",
    stemmer_type="snowball",
    ngram_range=None,
    language="portuguese",
    regex_pattern=r"\w+",
    custom_lemmatizer=None,
):
    """Pré-processamento configurável para experimentos de PLN.

    tokenizer   : "word" | "wordpunct" | "whitespace" | "regex"
    morphology  : "none" | "stemming" | "lemmatization"
    stemmer_type: "snowball" | "rslp"
    ngram_range : None | (1,1) | (1,2) | (1,3) | (2,2) ...
    """
    if lowercase:
        text = text.lower()

    if tokenizer == "word":
        tokens = word_tokenize(text, language=language)
    elif tokenizer == "wordpunct":
        tokens = wordpunct_tokenize(text)
    elif tokenizer == "whitespace":
        tokens = WhitespaceTokenizer().tokenize(text)
    elif tokenizer == "regex":
        tokens = RegexpTokenizer(regex_pattern).tokenize(text)
    else:
        raise ValueError(f"Tokenizador inválido: {tokenizer}")

    if remove_punctuation:
        tokens = [t for t in tokens if t.isalnum()]

    if remove_stopwords:
        stop_words = set(stopwords.words(language))
        tokens = [t for t in tokens if t not in stop_words]

    if morphology == "stemming":
        if stemmer_type == "snowball":
            stemmer = SnowballStemmer(language)
        elif stemmer_type == "rslp":
            if language != "portuguese":
                raise ValueError("RSLP é específico para português.")
            stemmer = RSLPStemmer()
        else:
            raise ValueError(f"Stemmer inválido: {stemmer_type}")
        tokens = [stemmer.stem(t) for t in tokens]
    elif morphology == "lemmatization":
        if custom_lemmatizer is not None:
            tokens = [custom_lemmatizer(t) for t in tokens]
        elif language == "english":
            lemmatizer = WordNetLemmatizer()
            tokens = [lemmatizer.lemmatize(t) for t in tokens]
        else:
            raise ValueError("Para português, forneça um custom_lemmatizer.")
    elif morphology != "none":
        raise ValueError(f"Normalização inválida: {morphology}")

    if ngram_range is not None:
        min_n, max_n = ngram_range
        resultado = []
        for n in range(min_n, max_n + 1):
            if n == 1:
                resultado.extend(tokens)
            else:
                resultado.extend("_".join(g) for g in ngrams(tokens, n))
        tokens = resultado

    return tokens


# ---- Observação importante para o domínio ----
# A remoção de stopwords do NLTK descarta palavras que carregam a intenção em português
# ("quando", "quanto", "quem", "qual", "não"). Mantemos a grade original, mas registramos
# essa hipótese: espera-se que as configurações SEM remoção de stopwords vençam.
print("Exemplo:", preprocess_text("Quando o projeto XPTO-104 termina?", remove_stopwords=False))
print("Com stopwords removidas:", preprocess_text("Quando o projeto XPTO-104 termina?", remove_stopwords=True))

Exemplo: ['quando', 'o', 'projeto', 'termina']
Com stopwords removidas: ['projeto', 'termina']


In [10]:
# ============================================================
# 4.2 Grade das 36 configurações (idêntica ao preprocess.ipynb)
# ============================================================
tokenizers = ["word", "regex"]
lowercase = True
remove_stopwords_options = [False, True]
morphologies = ["none", "stemming"]
stemmer_types = ["snowball", "rslp"]
ngram_ranges = [None, (1, 2), (1, 3)]

CONFIGS = {}
contador = 1
for tokenizer, remove_sw, morphology, ngram_range in product(
    tokenizers, remove_stopwords_options, morphologies, ngram_ranges
):
    stemmers = stemmer_types if morphology == "stemming" else [None]
    for stemmer_type in stemmers:
        nome = f"config_{contador:02d}"
        config = {
            "lowercase": lowercase,
            "tokenizer": tokenizer,
            "remove_stopwords": remove_sw,
            "remove_punctuation": True,
            "morphology": morphology,
            "ngram_range": ngram_range,
            "language": "portuguese",
        }
        if morphology == "stemming":
            config["stemmer_type"] = stemmer_type
        if tokenizer == "regex":
            config["regex_pattern"] = r"[^\W_]+"   # mantém acentuados
        CONFIGS[nome] = config
        contador += 1

print(f"Total de configurações: {len(CONFIGS)}")
display(pd.DataFrame(CONFIGS).T.head())

Total de configurações: 36


,lowercase,tokenizer,remove_stopwords,remove_punctuation,morphology,ngram_range,language,stemmer_type,regex_pattern
config_01,True,word,False,True,none,None,portuguese,NaN,NaN
config_02,True,word,False,True,none,"(1, 2)",portuguese,NaN,NaN
config_03,True,word,False,True,none,"(1, 3)",portuguese,NaN,NaN
config_04,True,word,False,True,stemming,None,portuguese,snowball,NaN
config_05,True,word,False,True,stemming,None,portuguese,rslp,NaN


In [11]:
# ============================================================
# 4.3 Seleção da melhor configuração (CV agrupada, somente no TREINO)
# ============================================================
import time

def identidade(tokens):        # analyzer para TF-IDF já tokenizado (precisa ser nomeada, não lambda)
    return tokens

def tokenizar_serie(textos, config):
    return [preprocess_text(t, **config) for t in textos]

y_tr_int = treino["intencao"].values
grupos_tr = treino["template_id"].values
cv = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)

resultados = []
cache_tokens = {}
t0 = time.time()

for nome, config in CONFIGS.items():
    tokens = tokenizar_serie(treino["texto"].values, config)
    cache_tokens[nome] = tokens

    for clf_nome, clf in [("LinearSVC", LinearSVC(C=1.0, random_state=SEED)),
                          ("MultinomialNB", MultinomialNB(alpha=0.2))]:
        f1s, accs = [], []
        for tr_idx, va_idx in cv.split(tokens, y_tr_int, groups=grupos_tr):
            vec = TfidfVectorizer(analyzer=identidade, min_df=1, sublinear_tf=True)
            Xtr = vec.fit_transform([tokens[i] for i in tr_idx])
            Xva = vec.transform([tokens[i] for i in va_idx])
            modelo = clf.__class__(**clf.get_params())
            modelo.fit(Xtr, y_tr_int[tr_idx])
            pred = modelo.predict(Xva)
            f1s.append(f1_score(y_tr_int[va_idx], pred, average="macro", zero_division=0))
            accs.append(accuracy_score(y_tr_int[va_idx], pred))

        resultados.append({
            "config": nome,
            "classificador": clf_nome,
            "tokenizer": config["tokenizer"],
            "stopwords": config["remove_stopwords"],
            "morfologia": config["morphology"],
            "stemmer": config.get("stemmer_type"),
            "ngramas": str(config["ngram_range"]),
            "f1_macro_cv": np.mean(f1s),
            "desvio": np.std(f1s),
            "acuracia_cv": np.mean(accs),
        })

df_grid = pd.DataFrame(resultados).sort_values("f1_macro_cv", ascending=False).reset_index(drop=True)
print(f"Grade concluída em {time.time()-t0:.0f}s — {len(df_grid)} combinações avaliadas\n")
display(df_grid.head(10).round(4))

MELHOR = df_grid.iloc[0]
MELHOR_CONFIG_NOME = MELHOR["config"]
MELHOR_CONFIG = CONFIGS[MELHOR_CONFIG_NOME]
print("\nMelhor configuração de pré-processamento:", MELHOR_CONFIG_NOME,
      "com", MELHOR["classificador"], f"(F1-macro CV = {MELHOR['f1_macro_cv']:.4f})")
print(json.dumps(MELHOR_CONFIG, ensure_ascii=False, indent=2))

Grade concluída em 25s — 72 combinações avaliadas



,config,classificador,tokenizer,stopwords,morfologia,stemmer,ngramas,f1_macro_cv,desvio,acuracia_cv
0,config_22,LinearSVC,regex,False,stemming,snowball,None,0.3532,0.0451,0.3931
1,config_04,LinearSVC,word,False,stemming,snowball,None,0.3489,0.0460,0.3922
2,config_33,LinearSVC,regex,True,stemming,snowball,"(1, 2)",0.3458,0.0385,0.3703
3,config_15,LinearSVC,word,True,stemming,snowball,"(1, 2)",0.3377,0.0405,0.3666
4,config_19,LinearSVC,regex,False,none,None,None,0.3374,0.0218,0.3813
5,config_01,LinearSVC,word,False,none,None,None,0.3354,0.0184,0.3829
6,config_05,LinearSVC,word,False,stemming,rslp,None,0.3351,0.0458,0.3794
7,config_06,LinearSVC,word,False,stemming,snowball,"(1, 2)",0.3336,0.0534,0.3821
8,config_24,LinearSVC,regex,False,stemming,snowball,"(1, 2)",0.3325,0.0474,0.3805
9,config_23,LinearSVC,regex,False,stemming,rslp,None,0.3321,0.0440,0.3774



Melhor configuração de pré-processamento: config_22 com LinearSVC (F1-macro CV = 0.3532)
{
  "lowercase": true,
  "tokenizer": "regex",
  "remove_stopwords": false,
  "remove_punctuation": true,
  "morphology": "stemming",
  "ngram_range": null,
  "language": "portuguese",
  "stemmer_type": "snowball",
  "regex_pattern": "[^\\W_]+"
}


In [12]:
# ============================================================
# 4.4 O que a grade ensina + tokenização final dos três splits
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, titulo in zip(axes,
                           ["stopwords", "morfologia", "ngramas"],
                           ["Remoção de stopwords", "Normalização morfológica", "N-gramas"]):
    df_grid.groupby(col)["f1_macro_cv"].mean().plot(kind="bar", ax=ax, color="#dd6b20")
    ax.set_title(titulo); ax.set_ylabel("F1-macro (CV)"); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

display(df_grid.groupby(["tokenizer", "stopwords", "morfologia"])["f1_macro_cv"]
        .mean().sort_values(ascending=False).round(4).to_frame())

# ---- Tokenização definitiva com a melhor configuração ----
tokens_treino = cache_tokens[MELHOR_CONFIG_NOME]
tokens_val = tokenizar_serie(validacao["texto"].values, MELHOR_CONFIG)
tokens_teste = tokenizar_serie(teste["texto"].values, MELHOR_CONFIG)

# TF-IDF ajustado SOMENTE no treino (evita vazamento de vocabulário/IDF)
tfidf = TfidfVectorizer(analyzer=identidade, min_df=1, sublinear_tf=True)
Xtf_treino = tfidf.fit_transform(tokens_treino)
Xtf_val = tfidf.transform(tokens_val)
Xtf_teste = tfidf.transform(tokens_teste)

y_treino_int, y_val_int, y_teste_int = treino["intencao"].values, validacao["intencao"].values, teste["intencao"].values
y_treino_mac, y_val_mac, y_teste_mac = treino["macrointencao"].values, validacao["macrointencao"].values, teste["macrointencao"].values

print("TF-IDF:", Xtf_treino.shape, Xtf_val.shape, Xtf_teste.shape,
      f"| vocabulário = {len(tfidf.vocabulary_)} termos (só do treino)")

f1_macro_cv
tokenizer stopwords morfologia             
word      False     stemming         0.2965
regex     False     stemming         0.2955
                    none             0.2942
word      False     none             0.2934
regex     True      stemming         0.2885
word      True      stemming         0.2852
regex     True      none             0.2770
word      True      none             0.2757

TF-IDF: (3062, 577) (1156, 577) (1135, 577) | vocabulário = 577 termos (só do treino)


## 5. Representações: Word2Vec, BERT e combinações híbridas

Três famílias de features, todas ajustadas **apenas com o treino**:

1. **TF‑IDF** sobre os tokens da melhor configuração — baseline lexical forte para intenção.
2. **Word2Vec** (gensim, Skip-gram) treinado do zero no corpus de treino; o vetor da frase é a média dos vetores das palavras **ponderada pelo IDF do treino**.
3. **BERT congelado** — embeddings de sentença de um modelo pré-treinado em português (BERTimbau) ou multilíngue, sem ajuste de pesos.
4. **Representações híbridas** — combinações de blocos normalizados em L2 e padronizados, incluindo o híbrido semântico `[Word2Vec ⊕ BERT]`, o híbrido lexical e o híbrido completo.

O benchmark completo selecionou **Híbrido W2V+BERT**. O Word2Vec representa relações distribucionais próprias do vocabulário do portfólio, enquanto o BERT congelado acrescenta contexto e conhecimento linguístico pré-treinado. A concatenação dos dois vetores foi superior às representações isoladas na execução de referência.

In [13]:
# ============================================================
# 5.1 Word2Vec treinado SOMENTE no corpus de treino
# ============================================================
W2V_DIM = 200

if W2V_OK:
    w2v = Word2Vec(
        sentences=tokens_treino,      # <- só o treino
        vector_size=W2V_DIM,
        window=5,
        min_count=1,
        sg=1,                         # skip-gram
        negative=10,
        epochs=60,
        workers=2,
        seed=SEED,
    )
    print("Vocabulário Word2Vec:", len(w2v.wv), "palavras")
    for palavra in ["prazo", "risco", "orcamento", "orçamento", "alerta", "relatorio", "relatório"]:
        if palavra in w2v.wv:
            viz = ", ".join(p for p, _ in w2v.wv.most_similar(palavra, topn=5))
            print(f"  {palavra:<12} ~ {viz}")

    idf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
    idf_padrao = float(np.max(tfidf.idf_))

    def vetor_w2v(tokens):
        vetores, pesos = [], []
        for t in tokens:
            if t in w2v.wv:
                vetores.append(w2v.wv[t])
                pesos.append(idf.get(t, idf_padrao))
        if not vetores:
            return np.zeros(W2V_DIM, dtype=np.float32)
        return np.average(np.vstack(vetores), axis=0, weights=pesos).astype(np.float32)

    Xw2v_treino = np.vstack([vetor_w2v(t) for t in tokens_treino])
    Xw2v_val = np.vstack([vetor_w2v(t) for t in tokens_val])
    Xw2v_teste = np.vstack([vetor_w2v(t) for t in tokens_teste])
    print("\nWord2Vec:", Xw2v_treino.shape, Xw2v_val.shape, Xw2v_teste.shape)

    oov_teste = np.mean([sum(t not in w2v.wv for t in toks) / max(1, len(toks)) for toks in tokens_teste])
    print(f"Taxa média de OOV no teste: {100*oov_teste:.1f}% (esperado > 0 — o teste tem templates inéditos)")
else:
    Xw2v_treino = Xw2v_val = Xw2v_teste = None
    print("gensim indisponível: etapa Word2Vec ignorada.")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Vocabulário Word2Vec: 577 palavras

Word2Vec: (3062, 200) (1156, 200) (1135, 200)
Taxa média de OOV no teste: 6.2% (esperado > 0 — o teste tem templates inéditos)


In [14]:
# ============================================================
# 5.2 Embeddings BERT congelados (sem fine-tuning)
#     Texto CRU — BERT tem tokenizador próprio; não usar a pipeline de stemming aqui.
# ============================================================
MODELO_BERT = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# Alternativa 100% PT-BR (mais pesada, sem treino contrastivo de sentença):
# MODELO_BERT = "neuralmind/bert-base-portuguese-cased"

try:
    import torch
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False
    print("PyTorch indisponível:", e)

def embeddings_sentence_transformers(textos, modelo):
    return modelo.encode(list(textos), batch_size=64, convert_to_numpy=True,
                         show_progress_bar=False, normalize_embeddings=True)

def embeddings_transformers(textos, tok, mdl, batch=32):
    saidas = []
    mdl.eval()
    with torch.no_grad():
        for i in range(0, len(textos), batch):
            lote = list(textos[i:i + batch])
            enc = tok(lote, padding=True, truncation=True, max_length=64, return_tensors="pt")
            out = mdl(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            media = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
            saidas.append(torch.nn.functional.normalize(media, dim=1).cpu().numpy())
    return np.vstack(saidas)

BERT_OK = TORCH_OK
try:
    if not TORCH_OK:
        raise RuntimeError("PyTorch ausente")
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer(MODELO_BERT)
    encode = lambda textos: embeddings_sentence_transformers(textos, st_model)
except Exception as e:
    print("sentence-transformers indisponível, usando transformers puro:", e)
    try:
        from transformers import AutoTokenizer, AutoModel
        _tok = AutoTokenizer.from_pretrained(MODELO_BERT)
        _mdl = AutoModel.from_pretrained(MODELO_BERT)
        encode = lambda textos: embeddings_transformers(np.array(textos), _tok, _mdl)
    except Exception as e2:
        BERT_OK = False
        print("BERT indisponível:", e2)

if BERT_OK:
    t0 = time.time()
    Xbert_treino = encode(treino["texto"].values)
    Xbert_val = encode(validacao["texto"].values)
    Xbert_teste = encode(teste["texto"].values)
    print(f"BERT ({MODELO_BERT}) em {time.time()-t0:.0f}s:",
          Xbert_treino.shape, Xbert_val.shape, Xbert_teste.shape)
else:
    Xbert_treino = Xbert_val = Xbert_teste = None

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

BERT (sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) em 3s: (3062, 384) (1156, 384) (1135, 384)


In [15]:
# ============================================================
# 5.3 Blocos de features e representações híbridas
#     Inclui W2V+BERT, combinações lexicais e a combinação completa
# ============================================================

# TF-IDF de caracteres: robusto a erro de digitação, abreviação e forma inédita de escrever
tfidf_char = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
Xch_treino = tfidf_char.fit_transform(treino["texto"].values)     # fit só no treino
Xch_val = tfidf_char.transform(validacao["texto"].values)
Xch_teste = tfidf_char.transform(teste["texto"].values)
print("TF-IDF de caracteres:", Xch_treino.shape)

SVD_DIM = min(200, Xtf_treino.shape[1] - 1)
svd = TruncatedSVD(n_components=SVD_DIM, random_state=SEED)
Xsvd_treino = svd.fit_transform(Xtf_treino)
Xsvd_val, Xsvd_teste = svd.transform(Xtf_val), svd.transform(Xtf_teste)

SVD_CHAR_DIM = min(250, Xch_treino.shape[1] - 1)
svd_char = TruncatedSVD(n_components=SVD_CHAR_DIM, random_state=SEED)
Xsvdc_treino = svd_char.fit_transform(Xch_treino)
Xsvdc_val, Xsvdc_teste = svd_char.transform(Xch_val), svd_char.transform(Xch_teste)
print(f"SVD palavras: {SVD_DIM} comps (var. {svd.explained_variance_ratio_.sum():.2f}) | "
      f"SVD caracteres: {SVD_CHAR_DIM} comps (var. {svd_char.explained_variance_ratio_.sum():.2f})")

SCALERS = {}   # guardados para reproduzir exatamente a mesma transformação na inferência

def bloco(nome, tr, va, te):
    """Normaliza L2 e padroniza um bloco denso (escalador ajustado no treino)."""
    if tr is None:
        return None
    sc = StandardScaler()
    saida = (sc.fit_transform(normalize(tr)), sc.transform(normalize(va)), sc.transform(normalize(te)))
    SCALERS[nome] = sc
    return saida

blocos = {
    "svd_palavra": bloco("svd_palavra", Xsvd_treino, Xsvd_val, Xsvd_teste),
    "svd_char": bloco("svd_char", Xsvdc_treino, Xsvdc_val, Xsvdc_teste),
}
if Xw2v_treino is not None:
    blocos["w2v"] = bloco("w2v", Xw2v_treino, Xw2v_val, Xw2v_teste)
if Xbert_treino is not None:
    blocos["bert"] = bloco("bert", Xbert_treino, Xbert_val, Xbert_teste)

ESPARSOS = {
    "tfidf_palavra": (Xtf_treino, Xtf_val, Xtf_teste),
    "tfidf_char": (Xch_treino, Xch_val, Xch_teste),
}

def montar(nomes):
    if len(nomes) == 1 and nomes[0] in ESPARSOS:
        return ESPARSOS[nomes[0]]
    return tuple(np.hstack([blocos[n][k] for n in nomes]) for k in range(3))

# Cada representação é declarada pelos blocos que a compõem (usado também na inferência)
REP_BLOCOS = {
    "TF-IDF palavras": ["tfidf_palavra"],
    "TF-IDF caracteres": ["tfidf_char"],
}
if "w2v" in blocos:
    REP_BLOCOS["Word2Vec"] = ["w2v"]
if "bert" in blocos:
    REP_BLOCOS["BERT congelado"] = ["bert"]
if "w2v" in blocos and "bert" in blocos:
    REP_BLOCOS["Híbrido W2V+BERT"] = ["w2v", "bert"]
REP_BLOCOS["Híbrido lexical (palavras+chars)"] = ["svd_palavra", "svd_char"]
REP_BLOCOS["Híbrido completo"] = [n for n in ["svd_palavra", "svd_char", "w2v", "bert"] if n in blocos]

REPRESENTACOES = {nome: montar(nomes) for nome, nomes in REP_BLOCOS.items()}

for nome, (tr, _, _) in REPRESENTACOES.items():
    print(f"{nome:<34} dim = {tr.shape[1]}")

TF-IDF de caracteres: (3062, 6209)


SVD palavras: 200 comps (var. 0.87) | SVD caracteres: 250 comps (var. 0.92)
TF-IDF palavras                    dim = 577
TF-IDF caracteres                  dim = 6209
Word2Vec                           dim = 200
BERT congelado                     dim = 384
Híbrido W2V+BERT                   dim = 584
Híbrido lexical (palavras+chars)   dim = 450
Híbrido completo                   dim = 1034


## 6. Treino e comparação das representações

Alvo principal: **intenção específica** (31 classes). A macrointenção é derivada do catálogo (`INTENCAO2MACRO`), o que garante coerência entre os dois níveis por construção.

Todos os modelos são escolhidos pela **validação**. O teste permanece fechado até a avaliação final. A tabela produzida pela célula seguinte é ordenada pelo F1-macro da intenção específica; portanto, a escolha do híbrido W2V+BERT decorre do benchmark, e não de uma preferência definida previamente.

In [16]:
# ============================================================
# 6.1 Comparação: representação x classificador (avaliado na VALIDAÇÃO)
# ============================================================
def avaliar(y_true_int, y_pred_int):
    macro_true = np.array([INTENCAO2MACRO[i] for i in y_true_int])
    macro_pred = np.array([INTENCAO2MACRO[i] for i in y_pred_int])
    return {
        "acc_intencao": accuracy_score(y_true_int, y_pred_int),
        "f1_intencao": f1_score(y_true_int, y_pred_int, average="macro", zero_division=0),
        "acc_macro": accuracy_score(macro_true, macro_pred),
        "f1_macro": f1_score(macro_true, macro_pred, average="macro", zero_division=0),
    }

CLASSIFICADORES = {
    "LogReg": lambda: LogisticRegression(max_iter=3000, C=5.0, class_weight="balanced", random_state=SEED),
    "LinearSVC": lambda: LinearSVC(C=1.0, class_weight="balanced", random_state=SEED),
}

linhas, modelos_treinados = [], {}
for nome_rep, (Xtr, Xva, Xte) in REPRESENTACOES.items():
    for nome_clf, fabrica in CLASSIFICADORES.items():
        clf = fabrica()
        clf.fit(Xtr, y_treino_int)
        pred = clf.predict(Xva)
        m = avaliar(y_val_int, pred)
        m.update({"representacao": nome_rep, "classificador": nome_clf})
        linhas.append(m)
        modelos_treinados[(nome_rep, nome_clf)] = clf

df_rep = (pd.DataFrame(linhas)[["representacao", "classificador", "acc_intencao",
                                "f1_intencao", "acc_macro", "f1_macro"]]
          .sort_values("f1_intencao", ascending=False).reset_index(drop=True))
display(df_rep.round(4))

fig, ax = plt.subplots(figsize=(11, 5))
piv = df_rep.pivot(index="representacao", columns="classificador", values="f1_intencao").sort_values("LogReg")
piv.plot(kind="barh", ax=ax, color=["#2b6cb0", "#dd6b20"])
ax.set_xlabel("F1-macro (intenção específica) — validação")
ax.set_title("Qual representação captura melhor a intenção?")
plt.tight_layout(); plt.show()

MELHOR_REP = df_rep.iloc[0]["representacao"]
MELHOR_CLF = df_rep.iloc[0]["classificador"]
print(f"Melhor combinação na validação: {MELHOR_REP} + {MELHOR_CLF} "
      f"(F1-macro intenção = {df_rep.iloc[0]['f1_intencao']:.4f})")

,representacao,classificador,acc_intencao,f1_intencao,acc_macro,f1_macro
0,Híbrido completo,LogReg,0.5415,0.5425,0.8036,0.8280
1,Híbrido W2V+BERT,LogReg,0.5355,0.5332,0.8529,0.8486
2,BERT congelado,LogReg,0.4905,0.4966,0.8391,0.8543
3,TF-IDF caracteres,LinearSVC,0.4801,0.4878,0.7370,0.7543
4,TF-IDF caracteres,LogReg,0.4784,0.4792,0.7517,0.7703
5,Híbrido completo,LinearSVC,0.4645,0.4745,0.6964,0.6887
6,Híbrido W2V+BERT,LinearSVC,0.4749,0.4670,0.7353,0.7396
7,TF-IDF palavras,LinearSVC,0.4628,0.4652,0.7673,0.7721
8,BERT congelado,LinearSVC,0.4567,0.4576,0.7656,0.7122
9,TF-IDF palavras,LogReg,0.4394,0.4535,0.7292,0.7373


Melhor combinação na validação: Híbrido completo + LogReg (F1-macro intenção = 0.5425)


In [17]:
# ============================================================
# 6.2 Por que o split agrupado importa: o mesmo modelo nos dois protocolos
# ============================================================
def experimento(split_col):
    tr = df[df[split_col] == "treino"]
    te = df[df[split_col] == "teste"]
    tok_tr = [preprocess_text(t, **MELHOR_CONFIG) for t in tr["texto"]]
    tok_te = [preprocess_text(t, **MELHOR_CONFIG) for t in te["texto"]]
    v = TfidfVectorizer(analyzer=identidade, min_df=1, sublinear_tf=True)
    Xtr, Xte = v.fit_transform(tok_tr), v.transform(tok_te)
    clf = LinearSVC(C=1.0, class_weight="balanced", random_state=SEED).fit(Xtr, tr["intencao"])
    pred = clf.predict(Xte)
    return avaliar(te["intencao"].values, pred)

m_ingenuo = experimento("split_ingenuo")
m_agrupado = experimento("split")

comp = pd.DataFrame({
    "split aleatório (COM vazamento de template)": m_ingenuo,
    "split agrupado por template (honesto)": m_agrupado,
}).T.round(4)
display(comp)

delta = m_ingenuo["f1_intencao"] - m_agrupado["f1_intencao"]
print(f"O split aleatório infla o F1-macro da intenção em {delta:+.4f} "
      f"({100*delta/max(m_agrupado['f1_intencao'], 1e-9):+.1f}%).")
print("Todos os números reportados adiante usam o protocolo agrupado.")

,acc_intencao,f1_intencao,acc_macro,f1_macro
split aleatório (COM vazamento de template),0.9963,0.9959,0.9991,0.9994
split agrupado por template (honesto),0.5859,0.5736,0.8150,0.7959


O split aleatório infla o F1-macro da intenção em +0.4223 (+73.6%).
Todos os números reportados adiante usam o protocolo agrupado.


## 7. Classificação hierárquica (macro → específica) vs. modelo plano

Duas arquiteturas para o mesmo problema:

- **Plano**: um classificador de 31 classes; a macrointenção sai do catálogo (`INTENCAO2MACRO`). Coerência garantida, mas todas as classes competem entre si.
- **Hierárquico**: um classificador de macrointenção (4 classes) + um especialista por macro. Erros de macro se propagam, mas cada especialista resolve um problema menor e mais homogêneo — e o roteamento do RF07 depende exatamente do acerto da macro.

A escolha entre os dois é feita pela validação. Na execução de referência, a arquitetura **hierárquica** foi escolhida e mantida no modelo final.

In [18]:
# ============================================================
# 7.1 Modelo hierárquico
# ============================================================
Xtr_best, Xva_best, Xte_best = REPRESENTACOES[MELHOR_REP]

def novo_clf():
    return LogisticRegression(max_iter=3000, C=5.0, class_weight="balanced", random_state=SEED)

class ClassificadorHierarquico:
    """Etapa 1: macrointenção. Etapa 2: especialista da macro predita."""

    def __init__(self, fabrica=novo_clf):
        self.fabrica = fabrica
        self.macro_clf = None
        self.especialistas = {}
        self.unica = {}

    def fit(self, X, y_macro, y_int):
        self.macro_clf = self.fabrica()
        self.macro_clf.fit(X, y_macro)
        for macro in np.unique(y_macro):
            mask = y_macro == macro
            classes = np.unique(y_int[mask])
            if len(classes) == 1:
                self.unica[macro] = classes[0]
            else:
                clf = self.fabrica()
                clf.fit(X[mask], y_int[mask])
                self.especialistas[macro] = clf
        return self

    def predict(self, X):
        macros = self.macro_clf.predict(X)
        preds = np.empty(len(macros), dtype=object)
        for macro in np.unique(macros):
            mask = macros == macro
            if macro in self.unica:
                preds[mask] = self.unica[macro]
            else:
                preds[mask] = self.especialistas[macro].predict(X[mask])
        return preds

    def predict_proba_intencao(self, X):
        """Probabilidade conjunta P(macro) * P(intenção | macro), na ordem de self.classes_."""
        p_macro = self.macro_clf.predict_proba(X)
        macros = self.macro_clf.classes_
        colunas, nomes = [], []
        for j, macro in enumerate(macros):
            if macro in self.unica:
                colunas.append(p_macro[:, j:j + 1]); nomes.append(self.unica[macro])
            else:
                esp = self.especialistas[macro]
                colunas.append(esp.predict_proba(X) * p_macro[:, j:j + 1])
                nomes.extend(list(esp.classes_))
        P = np.hstack(colunas)
        self.classes_ = np.array(nomes)
        return P

hier = ClassificadorHierarquico().fit(Xtr_best, y_treino_mac, y_treino_int)
pred_hier_val = hier.predict(Xva_best)
m_hier = avaliar(y_val_int, pred_hier_val)

plano = modelos_treinados[(MELHOR_REP, MELHOR_CLF)]
m_plano = avaliar(y_val_int, plano.predict(Xva_best))

comp_arq = pd.DataFrame({"Plano (31 classes)": m_plano, "Hierárquico (macro -> específica)": m_hier}).T.round(4)
display(comp_arq)

USAR_HIERARQUICO = m_hier["f1_intencao"] >= m_plano["f1_intencao"]
print("Arquitetura escolhida pela validação:",
      "HIERÁRQUICA" if USAR_HIERARQUICO else "PLANA")

,acc_intencao,f1_intencao,acc_macro,f1_macro
Plano (31 classes),0.5415,0.5425,0.8036,0.8280
Hierárquico (macro -> específica),0.6306,0.6282,0.9196,0.9037


Arquitetura escolhida pela validação: HIERÁRQUICA


## 8. Refinamento do modelo

Três ajustes, todos decididos **sem tocar no teste**:

1. **Hiperparâmetros** — busca em grade com `PredefinedSplit`: o treino é o fold de ajuste e a validação é o único fold de avaliação. Nenhum exemplo de validação entra no ajuste dos parâmetros.
2. **Reajuste final** — com os hiperparâmetros escolhidos, o classificador é retreinado em **treino + validação** (os transformadores de features continuam ajustados só no treino, o que é conservador e mantém o teste intocado).
3. **Limiar de confiança** — abaixo do limiar o agente responde "não tenho informação suficiente" em vez de forçar uma intenção (RF08). O limiar é calibrado na validação pela curva risco × cobertura.

Na execução de referência, a busca escolheu `C=5.0` e `class_weight=balanced`; os limiares resultantes foram **0,925 para macrointenção** e **0,950 para intenção específica**. São valores deliberadamente conservadores: reduzem o risco de acionar um fluxo incorreto, mas também podem elevar a quantidade de abstenções e encaminhamentos genéricos.

In [19]:
# ============================================================
# 8.1 Busca de hiperparâmetros com PredefinedSplit (treino ajusta, validação avalia)
# ============================================================
from functools import partial
from scipy import sparse

def empilhar(A, B):
    return sparse.vstack([A, B]) if sparse.issparse(A) else np.vstack([A, B])

X_trval = empilhar(Xtr_best, Xva_best)
y_trval_int = np.concatenate([y_treino_int, y_val_int])
y_trval_mac = np.concatenate([y_treino_mac, y_val_mac])
test_fold = np.concatenate([-np.ones(len(y_treino_int), dtype=int), np.zeros(len(y_val_int), dtype=int)])
ps = PredefinedSplit(test_fold)

grade = {
    "C": [0.5, 1.0, 2.0, 5.0, 10.0, 20.0],
    "class_weight": [None, "balanced"],
}
busca = GridSearchCV(
    LogisticRegression(max_iter=4000, random_state=SEED),
    grade, scoring="f1_macro", cv=ps, n_jobs=-1, refit=False,
)
busca.fit(X_trval, y_trval_int)

df_busca = (pd.DataFrame(busca.cv_results_)[["param_C", "param_class_weight", "mean_test_score"]]
            .sort_values("mean_test_score", ascending=False).reset_index(drop=True))
display(df_busca.head(8).round(4))
print("Melhores hiperparâmetros:", busca.best_params_, "| F1-macro validação =", round(busca.best_score_, 4))

# ---- Modelo final: retreinado em treino + validação ----
if USAR_HIERARQUICO:
    # `partial` preserva a fábrica no artefato sem introduzir uma lambda não serializável.
    fabrica_final = partial(LogisticRegression, max_iter=4000, random_state=SEED,
                            **busca.best_params_)
    modelo_final = ClassificadorHierarquico(fabrica_final).fit(X_trval, y_trval_mac, y_trval_int)
    nome_arquitetura = "hierárquica"
else:
    modelo_final = LogisticRegression(max_iter=4000, random_state=SEED, **busca.best_params_)
    modelo_final.fit(X_trval, y_trval_int)
    nome_arquitetura = "plana"

print(f"Modelo final ({nome_arquitetura}) treinado em {X_trval.shape[0]} exemplos "
      f"({len(y_treino_int)} treino + {len(y_val_int)} validação).")

,param_C,param_class_weight,mean_test_score
0,0.5,balanced,0.5591
1,0.5,None,0.5502
2,1.0,None,0.5483
3,2.0,None,0.5446
4,1.0,balanced,0.5445
5,10.0,None,0.5429
6,5.0,balanced,0.5425
7,2.0,balanced,0.5424


Melhores hiperparâmetros: {'C': 0.5, 'class_weight': 'balanced'} | F1-macro validação = 0.5591


Modelo final (hierárquica) treinado em 4218 exemplos (3062 treino + 1156 validação).


In [20]:
# ============================================================
# 8.2 Limiares de confiança — calibrados na validação com um modelo treinado SÓ no treino
#     LIMIAR_MACRO  : decide se o pedido é roteado (RF07) ou recebe resposta de limitação (RF08)
#     LIMIAR_INTENCAO: decide se o handler específico é acionado ou se fica no fluxo genérico da macro
# ============================================================
def proba_intencao(modelo, X):
    if isinstance(modelo, ClassificadorHierarquico):
        P = modelo.predict_proba_intencao(X)
        return modelo.classes_, P
    return modelo.classes_, modelo.predict_proba(X)

def proba_macro(modelo, X):
    """Probabilidade por macrointenção (soma das intenções da macro, no caso plano)."""
    if isinstance(modelo, ClassificadorHierarquico):
        return modelo.macro_clf.classes_, modelo.macro_clf.predict_proba(X)
    classes, P = proba_intencao(modelo, X)
    macros = np.array(sorted({INTENCAO2MACRO[c] for c in classes}))
    M = np.zeros((P.shape[0], len(macros)))
    for j, m in enumerate(macros):
        cols = [k for k, c in enumerate(classes) if INTENCAO2MACRO[c] == m]
        M[:, j] = P[:, cols].sum(axis=1)
    return macros, M

fabrica_cal = partial(LogisticRegression, max_iter=4000, random_state=SEED,
                      **busca.best_params_)
if USAR_HIERARQUICO:
    modelo_cal = ClassificadorHierarquico(fabrica_cal).fit(Xtr_best, y_treino_mac, y_treino_int)
else:
    modelo_cal = fabrica_cal().fit(Xtr_best, y_treino_int)

classes_cal, P_val = proba_intencao(modelo_cal, Xva_best)
conf_val_int = P_val.max(axis=1)
pred_val = classes_cal[P_val.argmax(axis=1)]
acerto_val_int = (pred_val == y_val_int)

macros_cal, PM_val = proba_macro(modelo_cal, Xva_best)
conf_val_mac = PM_val.max(axis=1)
pred_val_mac = macros_cal[PM_val.argmax(axis=1)]
acerto_val_mac = (pred_val_mac == y_val_mac)

limiares = np.arange(0.05, 0.96, 0.025)

def curva(conf, acerto):
    linhas = []
    for lim in limiares:
        aceito = conf >= lim
        linhas.append({"limiar": lim,
                       "cobertura": aceito.mean(),
                       "acuracia_nos_aceitos": acerto[aceito].mean() if aceito.any() else np.nan})
    return pd.DataFrame(linhas)

def escolher_limiar(df_c, alvo_acuracia, cobertura_minima, custo_erro=2.0):
    """Menor limiar que atinge a acurácia alvo mantendo a cobertura mínima.

    Se nenhum atende, cai numa regra de utilidade: um acerto vale +1, um erro
    vale -custo_erro e uma abstenção vale 0. A utilidade esperada por pedido é
    cobertura * (acc*(1+custo) - custo) — ou seja, só vale responder enquanto a
    fatia marginal de pedidos aceitos acertar mais de custo/(1+custo) das vezes.
    """
    cand = df_c[(df_c.acuracia_nos_aceitos >= alvo_acuracia) & (df_c.cobertura >= cobertura_minima)]
    if len(cand):
        return float(cand.iloc[0]["limiar"]), True
    utilidade = df_c.cobertura * (df_c.acuracia_nos_aceitos * (1 + custo_erro) - custo_erro)
    return float(df_c.loc[utilidade.idxmax(), "limiar"]), False

curva_mac, curva_int = curva(conf_val_mac, acerto_val_mac), curva(conf_val_int, acerto_val_int)
LIMIAR_MACRO, ok_mac = escolher_limiar(curva_mac, alvo_acuracia=0.97, cobertura_minima=0.80)
LIMIAR_INTENCAO, ok_int = escolher_limiar(curva_int, alvo_acuracia=0.80, cobertura_minima=0.50)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, dfc, lim, titulo in [(axes[0], curva_mac, LIMIAR_MACRO, "Macrointenção (roteamento)"),
                             (axes[1], curva_int, LIMIAR_INTENCAO, "Intenção específica (handler)")]:
    ax.plot(dfc.limiar, dfc.cobertura, label="cobertura", color="#2b6cb0")
    ax.plot(dfc.limiar, dfc.acuracia_nos_aceitos, label="acurácia entre os aceitos", color="#38a169")
    ax.axvline(lim, ls="--", color="red", label=f"limiar = {lim:.3f}")
    ax.set_title(f"Risco × cobertura — {titulo}"); ax.set_xlabel("limiar"); ax.legend()
plt.tight_layout(); plt.show()

print(f"LIMIAR_MACRO    = {LIMIAR_MACRO:.3f}  (alvo de 97% de acerto atingido: {ok_mac})")
print(curva_mac[np.isclose(curva_mac.limiar, LIMIAR_MACRO)].round(4).to_string(index=False))
print(f"\nLIMIAR_INTENCAO = {LIMIAR_INTENCAO:.3f}  (alvo de 80% de acerto atingido: {ok_int})")
print(curva_int[np.isclose(curva_int.limiar, LIMIAR_INTENCAO)].round(4).to_string(index=False))

LIMIAR_MACRO    = 0.550  (alvo de 97% de acerto atingido: False)
 limiar  cobertura  acuracia_nos_aceitos
   0.55      0.974                0.9316

LIMIAR_INTENCAO = 0.700  (alvo de 80% de acerto atingido: True)
 limiar  cobertura  acuracia_nos_aceitos
    0.7      0.577                0.8066


## 9. Avaliação final no conjunto de teste

O teste é aberto **uma única vez**, agora. Ele contém apenas templates que nenhum modelo, nenhuma configuração de pré-processamento e nenhum hiperparâmetro chegou a ver — é a estimativa controlada de como o classificador se comporta diante de uma forma de pedir inédita.

### 9.1 Leitura dos resultados atuais

O híbrido W2V+BERT hierárquico atingiu, no teste, **0,6740 de acurácia e 0,6777 de F1-macro nas intenções específicas**. No nível de macrointenção, alcançou **0,9383 de acurácia e 0,9379 de F1-macro**, resultado mais forte e compatível com seu papel de roteamento inicial. Na validação, os valores foram 0,5666/0,5665 para intenção e 0,9083/0,8885 para macro.

A diferença entre validação e teste não deve ser interpretada como garantia de melhora: os conjuntos contêm grupos de templates diferentes e são relativamente pequenos. O dado mais seguro é que a macrointenção está mais madura que a intenção específica; esta última ainda erra aproximadamente um terço dos exemplos de teste antes da aplicação dos limiares.

### 9.2 Limitações do resultado

- Os prompts são majoritariamente sintéticos e derivados de templates; o split agrupado reduz vazamento de redação, mas não substitui um teste externo com solicitações reais.
- O Word2Vec é treinado em apenas 3.062 exemplos de treino, o que limita a cobertura semântica do vocabulário do domínio.
- O BERT é usado como extrator congelado, sem fine-tuning supervisionado para as intenções do projeto.
- Os limiares de 0,925 e 0,950 priorizam segurança, mas a cobertura após rejeição deve ser medida e acompanhada em uso real.
- A variação entre validação e teste indica sensibilidade ao conjunto avaliado; novas rodadas com dados reais e análise por classe são necessárias antes de considerar o modelo pronto para produção.

In [21]:
# ============================================================
# 9.3 Métricas finais (teste)
# ============================================================
classes_fin, P_teste = proba_intencao(modelo_final, Xte_best)
conf_teste = P_teste.max(axis=1)
pred_teste_int = classes_fin[P_teste.argmax(axis=1)]
pred_teste_mac = np.array([INTENCAO2MACRO[i] for i in pred_teste_int])

m_teste = avaliar(y_teste_int, pred_teste_int)
resumo = pd.DataFrame({
    "Validação (modelo treinado só no treino)": avaliar(y_val_int, pred_val),
    "TESTE (modelo final)": m_teste,
}).T.round(4)
display(resumo)

print("\n" + "=" * 72)
print("MACROINTENÇÃO — teste")
print("=" * 72)
print(classification_report(y_teste_mac, pred_teste_mac, digits=3, zero_division=0))

print("=" * 72)
print("INTENÇÃO ESPECÍFICA — teste")
print("=" * 72)
print(classification_report(y_teste_int, pred_teste_int, digits=3, zero_division=0))

,acc_intencao,f1_intencao,acc_macro,f1_macro
Validação (modelo treinado só no treino),0.6332,0.6270,0.9135,0.9115
TESTE (modelo final),0.7383,0.7401,0.9551,0.9477



MACROINTENÇÃO — teste
                precision    recall  f1-score   support

        ALERTA      1.000     0.946     0.972       202
      CONSULTA      0.940     0.963     0.952       492
FORA_DE_ESCOPO      0.932     0.891     0.911        46
     TRANSACAO      0.955     0.957     0.956       395

      accuracy                          0.955      1135
     macro avg      0.957     0.939     0.948      1135
  weighted avg      0.956     0.955     0.955      1135

INTENÇÃO ESPECÍFICA — teste
                                 precision    recall  f1-score   support

alerta_configurar_monitoramento      0.694     1.000     0.819        34
    alerta_desatualizacao_dados      0.967     0.853     0.906        34
       alerta_estouro_orcamento      1.000     0.548     0.708        42
     alerta_gerenciar_inscricao      0.882     0.968     0.923        31
            alerta_prazo_atraso      0.857     0.686     0.762        35
           alerta_risco_critico      0.963     1.000     0.

In [22]:
# ============================================================
# 9.4 Matrizes de confusão
# ============================================================
def plot_confusao(y_true, y_pred, labels, titulo, figsize=(7, 6), fonte=9, normalizar_linha=True):
    cm = confusion_matrix(y_true, y_pred, labels=labels).astype(float)
    if normalizar_linha:
        soma = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, soma, out=np.zeros_like(cm), where=soma > 0)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=fonte)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=fonte)
    ax.set_xlabel("predito"); ax.set_ylabel("verdadeiro"); ax.set_title(titulo)
    if len(labels) <= 8:
        for i in range(len(labels)):
            for j in range(len(labels)):
                ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center",
                        color="white" if cm[i, j] > 0.5 else "black", fontsize=fonte)
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()

plot_confusao(y_teste_mac, pred_teste_mac, MACROS, "Macrointenção — teste (normalizada por linha)")

ordem_int = [i for m in MACROS for i in CATALOGO[m]["intencoes"]]
plot_confusao(y_teste_int, pred_teste_int, ordem_int,
              "Intenção específica — teste (normalizada por linha)", figsize=(12, 11), fonte=7)

# ---- Confusões mais frequentes ----
erros = pd.DataFrame({"verdadeiro": y_teste_int, "predito": pred_teste_int})
erros = erros[erros.verdadeiro != erros.predito]
top_conf = (erros.groupby(["verdadeiro", "predito"]).size()
            .sort_values(ascending=False).head(15).rename("erros").reset_index())
top_conf["mesma_macro"] = top_conf.apply(
    lambda r: INTENCAO2MACRO[r["verdadeiro"]] == INTENCAO2MACRO[r["predito"]], axis=1)
print(f"{len(erros)} erros de intenção em {len(y_teste_int)} exemplos "
      f"({100*len(erros)/len(y_teste_int):.1f}%) — "
      f"{100*top_conf['mesma_macro'].mean():.0f}% das confusões mais comuns ficam dentro da mesma macro")
display(top_conf)

297 erros de intenção em 1135 exemplos (26.2%) — 87% das confusões mais comuns ficam dentro da mesma macro


,verdadeiro,predito,erros,mesma_macro
0,analise_tendencia_desempenho,consulta_cronograma_prazo,20,True
1,consulta_documentos_fonte,consulta_responsavel_equipe,14,True
2,alerta_estouro_orcamento,alerta_configurar_monitoramento,14,True
3,transacao_solicitar_atualizacao,transacao_exportar_dados,14,True
4,transacao_atualizar_cronograma,transacao_solicitar_atualizacao,13,True
5,transacao_exportar_dados,transacao_gerar_relatorio,13,True
6,consulta_documentos_fonte,consulta_orcamento_custo,12,True
7,comparacao_entre_projetos,transacao_atualizar_status,11,False
8,consulta_orcamento_custo,comparacao_entre_projetos,11,True
9,transacao_registrar_pendencia,transacao_cadastrar_projeto,10,True


In [23]:
# ============================================================
# 9.5 Efeito dos limiares no teste (RF07 roteamento / RF08 limitação)
# ============================================================
macros_fin, PM_teste = proba_macro(modelo_final, Xte_best)
conf_teste_mac = PM_teste.max(axis=1)
pred_teste_mac_lim = macros_fin[PM_teste.argmax(axis=1)]

aceito_mac = conf_teste_mac >= LIMIAR_MACRO
aceito_int = (conf_teste >= LIMIAR_INTENCAO) & aceito_mac

resumo_lim = pd.DataFrame([
    {"decisão": "roteamento por macrointenção",
     "limiar": round(LIMIAR_MACRO, 3),
     "cobertura": aceito_mac.mean(),
     "acurácia entre os aceitos": (pred_teste_mac_lim[aceito_mac] == y_teste_mac[aceito_mac]).mean() if aceito_mac.any() else np.nan},
    {"decisão": "handler da intenção específica",
     "limiar": round(LIMIAR_INTENCAO, 3),
     "cobertura": aceito_int.mean(),
     "acurácia entre os aceitos": (pred_teste_int[aceito_int] == y_teste_int[aceito_int]).mean() if aceito_int.any() else np.nan},
]).round(4)
display(resumo_lim)

print(f"{(~aceito_mac).sum()} pedidos ({100*(1-aceito_mac.mean()):.1f}%) receberiam a resposta de limitação.")
print(f"{(aceito_mac & ~aceito_int).sum()} pedidos ({100*(aceito_mac & ~aceito_int).mean():.1f}%) seriam roteados pela macro "
      f"mas tratados pelo fluxo genérico, sem acionar um handler específico de baixa confiança.")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(conf_teste[pred_teste_int == y_teste_int], bins=30, alpha=0.75, label="acertos", color="#38a169")
axes[0].hist(conf_teste[pred_teste_int != y_teste_int], bins=30, alpha=0.75, label="erros", color="#e53e3e")
axes[0].axvline(LIMIAR_INTENCAO, ls="--", color="black", label=f"limiar {LIMIAR_INTENCAO:.2f}")
axes[0].set_title("Confiança na intenção específica (teste)"); axes[0].set_xlabel("P(máx)"); axes[0].legend()

cob, accs = [], []
for lim in limiares:
    a = conf_teste >= lim
    cob.append(a.mean())
    accs.append((pred_teste_int[a] == y_teste_int[a]).mean() if a.any() else np.nan)
axes[1].plot(cob, accs, marker="o", ms=3, color="#805ad5")
axes[1].set_xlabel("cobertura"); axes[1].set_ylabel("acurácia entre os aceitos")
axes[1].set_title("Curva risco × cobertura (teste)")
plt.tight_layout(); plt.show()

,decisão,limiar,cobertura,acurácia entre os aceitos
0,roteamento por macrointenção,0.55,0.9877,0.9634
1,handler da intenção específica,0.70,0.7189,0.8775


14 pedidos (1.2%) receberiam a resposta de limitação.
305 pedidos (26.9%) seriam roteados pela macro mas tratados pelo fluxo genérico, sem acionar um handler específico de baixa confiança.


## 10. Inferência: do texto ao roteamento (RF01 → RF02 → RF03 → RF07)

`classificar(texto)` devolve exatamente o que o `PipelinePLN` precisa: a **macrointenção** (que define o fluxo), a **intenção específica** (que define o handler), a **confiança** e as alternativas — ou a resposta de limitação quando a confiança fica abaixo do limiar calibrado.

In [24]:
# ============================================================
# 10.1 Pipeline de inferência
# ============================================================
def vetorizar(textos):
    """Aplica exatamente as mesmas transformações ajustadas no treino."""
    textos = list(textos)
    toks = [preprocess_text(t, **MELHOR_CONFIG) for t in textos]
    nomes = REP_BLOCOS[MELHOR_REP]

    if nomes == ["tfidf_palavra"]:
        return tfidf.transform(toks)
    if nomes == ["tfidf_char"]:
        return tfidf_char.transform(textos)

    partes = []
    for n in nomes:
        if n == "svd_palavra":
            partes.append(SCALERS[n].transform(normalize(svd.transform(tfidf.transform(toks)))))
        elif n == "svd_char":
            partes.append(SCALERS[n].transform(normalize(svd_char.transform(tfidf_char.transform(textos)))))
        elif n == "w2v":
            partes.append(SCALERS[n].transform(normalize(np.vstack([vetor_w2v(t) for t in toks]))))
        elif n == "bert":
            partes.append(SCALERS[n].transform(normalize(encode(textos))))
    return np.hstack(partes) if len(partes) > 1 else partes[0]


RESPOSTA_LIMITACAO = ("Não consegui identificar com segurança o que você precisa dentro do portfólio. "
                      "Pode reformular indicando o projeto e a informação desejada?")

def classificar(texto, top_k=3):
    X = vetorizar([texto])
    classes, P = proba_intencao(modelo_final, X)
    macros, PM = proba_macro(modelo_final, X)

    p, pm = P[0], PM[0]
    ordem = np.argsort(p)[::-1][:top_k]
    macro = macros[int(np.argmax(pm))]
    conf_macro = float(pm.max())
    intencao = classes[ordem[0]]
    conf_int = float(p[ordem[0]])

    roteia = conf_macro >= LIMIAR_MACRO
    handler_especifico = roteia and conf_int >= LIMIAR_INTENCAO and macro != "FORA_DE_ESCOPO"

    return {
        "texto": texto,
        "macrointencao": macro if roteia else "INDEFINIDA",
        "confianca_macro": round(conf_macro, 4),
        "intencao": intencao if handler_especifico else None,
        "confianca_intencao": round(conf_int, 4),
        "fluxo": (CATALOGO[macro]["fluxo"] if roteia else "Resposta padrão de limitação (RF08)"),
        "handler": ("handler:" + intencao if handler_especifico
                    else ("fluxo genérico de " + macro if roteia else "limitacao")),
        "alternativas": [(str(classes[i]), round(float(p[i]), 4)) for i in ordem],
        "resposta_sugerida": None if roteia else RESPOSTA_LIMITACAO,
    }


exemplos = [
    "quanto já foi gasto na obra da Linha 6?",
    "me avisa se o projeto de sinalização atrasar mais de uma semana",
    "prorroga o prazo do PRJ-0442 até março",
    "compare a execução orçamentária da Linha 2 com a da Linha 15",
    "gera um relatório executivo do subportfólio de modernização",
    "quem é que responde pelo pátio novo?",
    "bom dia, tudo certo?",
    "qual a previsão do tempo para amanhã?",
    "os riscos do programa de acessibilidade foram revisados?",
    "manda pro Carlos a cobrança da planilha de marcos",
    "abcdefg xyz",
]

for e in exemplos:
    r = classificar(e)
    print(f"[{r['macrointencao']:<14} {r['confianca_macro']:.2f}] {e}")
    print(f"    -> {r['handler']} (conf. intenção = {r['confianca_intencao']:.2f})")
    print(f"       alternativas: {r['alternativas']}")

[CONSULTA       1.00] quanto já foi gasto na obra da Linha 6?
    -> handler:consulta_orcamento_custo (conf. intenção = 1.00)
       alternativas: [('consulta_orcamento_custo', 0.9978), ('consulta_visao_portfolio', 0.0006), ('analise_tendencia_desempenho', 0.0003)]
[ALERTA         1.00] me avisa se o projeto de sinalização atrasar mais de uma semana
    -> handler:alerta_prazo_atraso (conf. intenção = 1.00)
       alternativas: [('alerta_prazo_atraso', 0.9999), ('alerta_gerenciar_inscricao', 0.0001), ('alerta_desatualizacao_dados', 0.0)]
[CONSULTA       0.74] prorroga o prazo do PRJ-0442 até março
    -> handler:consulta_cronograma_prazo (conf. intenção = 0.73)
       alternativas: [('consulta_cronograma_prazo', 0.7321), ('transacao_atualizar_cronograma', 0.1417), ('alerta_prazo_atraso', 0.0589)]
[CONSULTA       1.00] compare a execução orçamentária da Linha 2 com a da Linha 15
    -> handler:comparacao_entre_projetos (conf. intenção = 0.98)
       alternativas: [('comparacao_entre_pro

[CONSULTA       1.00] os riscos do programa de acessibilidade foram revisados?
    -> handler:consulta_riscos_projeto (conf. intenção = 1.00)
       alternativas: [('consulta_riscos_projeto', 0.9997), ('consulta_escopo_tap', 0.0001), ('alerta_risco_critico', 0.0001)]
[TRANSACAO      1.00] manda pro Carlos a cobrança da planilha de marcos
    -> handler:transacao_solicitar_atualizacao (conf. intenção = 0.98)
       alternativas: [('transacao_solicitar_atualizacao', 0.9778), ('transacao_exportar_dados', 0.022), ('transacao_registrar_pendencia', 0.0001)]
[FORA_DE_ESCOPO 0.97] abcdefg xyz
    -> fluxo genérico de FORA_DE_ESCOPO (conf. intenção = 0.85)
       alternativas: [('fora_escopo_social', 0.8522), ('fora_escopo_externo', 0.1144), ('transacao_cadastrar_projeto', 0.0142)]


In [25]:
# ============================================================
# 10.2 Exportação dos artefatos (dataset, catálogo, modelo, métricas)
# ============================================================
import joblib

os.makedirs("artefatos", exist_ok=True)

df.drop(columns=["chave"]).to_csv("artefatos/dataset_intencoes.csv", index=False, encoding="utf-8")
df_catalogo.to_csv("artefatos/catalogo_intencoes.csv", index=False, encoding="utf-8")

with open("artefatos/catalogo_intencoes.json", "w", encoding="utf-8") as f:
    json.dump(CATALOGO, f, ensure_ascii=False, indent=2)

metricas = {
    "melhor_config_preprocessamento": {"nome": MELHOR_CONFIG_NOME, **{k: str(v) for k, v in MELHOR_CONFIG.items()}},
    "melhor_representacao": MELHOR_REP,
    "arquitetura": nome_arquitetura,
    "hiperparametros": {k: str(v) for k, v in busca.best_params_.items()},
    "limiar_macro": LIMIAR_MACRO,
    "limiar_intencao": LIMIAR_INTENCAO,
    "teste": {k: round(float(v), 4) for k, v in m_teste.items()},
    "validacao": {k: round(float(v), 4) for k, v in avaliar(y_val_int, pred_val).items()},
    "tamanhos": {"treino": len(treino), "validacao": len(validacao), "teste": len(teste)},
    "protocolo": "split agrupado por template_id; transformadores ajustados apenas no treino",
}
with open("artefatos/metricas.json", "w", encoding="utf-8") as f:
    json.dump(metricas, f, ensure_ascii=False, indent=2)

# Obs.: ao recarregar o .joblib em outro processo, as funções `identidade`,
# `preprocess_text` e a classe `ClassificadorHierarquico` precisam estar definidas
# no ambiente (o pickle guarda a referência, não o código).
try:
    joblib.dump({
        "tfidf_palavras": tfidf,
        "tfidf_caracteres": tfidf_char,
        "svd_palavras": svd,
        "svd_caracteres": svd_char,
        "scalers": SCALERS,
        "modelo": modelo_final,
        "config_preprocessamento": MELHOR_CONFIG,
        "representacao": MELHOR_REP,
        "blocos_da_representacao": REP_BLOCOS[MELHOR_REP],
        "limiar_macro": LIMIAR_MACRO,
        "limiar_intencao": LIMIAR_INTENCAO,
        "intencao2macro": INTENCAO2MACRO,
    }, "artefatos/classificador_intencoes.joblib")
    print("Modelo serializado em artefatos/classificador_intencoes.joblib")
except Exception as e:
    print("Não foi possível serializar o modelo neste ambiente:", e)

if W2V_OK:
    w2v.save("artefatos/word2vec_portfolio.model")

print(json.dumps(metricas, ensure_ascii=False, indent=2))
print("\nArquivos gerados:")
for f_ in sorted(os.listdir("artefatos")):
    print(" -", f_, f"({os.path.getsize(os.path.join('artefatos', f_))/1024:.0f} KB)")

# Para baixar tudo do Colab, descomente as três linhas abaixo:
# !zip -qr artefatos.zip artefatos
# from google.colab import files
# files.download("artefatos.zip")

Modelo serializado em artefatos/classificador_intencoes.joblib
{
  "melhor_config_preprocessamento": {
    "nome": "config_22",
    "lowercase": "True",
    "tokenizer": "regex",
    "remove_stopwords": "False",
    "remove_punctuation": "True",
    "morphology": "stemming",
    "ngram_range": "None",
    "language": "portuguese",
    "stemmer_type": "snowball",
    "regex_pattern": "[^\\W_]+"
  },
  "melhor_representacao": "Híbrido completo",
  "arquitetura": "hierárquica",
  "hiperparametros": {
    "C": "0.5",
    "class_weight": "balanced"
  },
  "limiar_macro": 0.5500000000000003,
  "limiar_intencao": 0.7000000000000003,
  "teste": {
    "acc_intencao": 0.7383,
    "f1_intencao": 0.7401,
    "acc_macro": 0.9551,
    "f1_macro": 0.9477
  },
  "validacao": {
    "acc_intencao": 0.6332,
    "f1_intencao": 0.627,
    "acc_macro": 0.9135,
    "f1_macro": 0.9115
  },
  "tamanhos": {
    "treino": 3062,
    "validacao": 1156,
    "teste": 1135
  },
  "protocolo": "split agrupado por temp

## 11. (Opcional) Fine-tuning do BERTimbau — exige GPU

A célula abaixo ajusta os pesos do `neuralmind/bert-base-portuguese-cased` para as 31 intenções, usando **os mesmos splits agrupados** (nenhum template do teste entra no treino). Mude `RODAR_FINETUNING = True` com GPU ativada (`Ambiente de execução → Alterar o tipo de ambiente → T4`). Em CPU leva muito tempo — por isso vem desligada.

Compare o resultado com o da matriz híbrida: em datasets de intenção pequenos e bem estruturados como este, o ganho do fine-tuning costuma ser modesto e o custo operacional bem maior — é essa comparação que justifica a escolha de arquitetura no relatório.

In [26]:
# ============================================================
# 11.1 Fine-tuning do BERTimbau (opcional)
# ============================================================
RODAR_FINETUNING = False        # <- mude para True com GPU

if RODAR_FINETUNING:
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer)
    from torch.utils.data import Dataset as TorchDataset

    MODELO_FT = "neuralmind/bert-base-portuguese-cased"
    rotulos = sorted(INTENCOES)
    rot2id = {r: i for i, r in enumerate(rotulos)}

    tok_ft = AutoTokenizer.from_pretrained(MODELO_FT)

    class DsIntencoes(TorchDataset):
        def __init__(self, textos, rotulos_y):
            self.enc = tok_ft(list(textos), truncation=True, padding="max_length", max_length=48)
            self.y = [rot2id[r] for r in rotulos_y]
        def __len__(self):
            return len(self.y)
        def __getitem__(self, i):
            item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
            item["labels"] = torch.tensor(self.y[i])
            return item

    ds_tr = DsIntencoes(treino["texto"].values, y_treino_int)
    ds_va = DsIntencoes(validacao["texto"].values, y_val_int)
    ds_te = DsIntencoes(teste["texto"].values, y_teste_int)

    modelo_ft = AutoModelForSequenceClassification.from_pretrained(
        MODELO_FT, num_labels=len(rotulos),
        id2label={i: r for r, i in rot2id.items()}, label2id=rot2id,
    )

    def metricas_ft(eval_pred):
        logits, y = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {"acc": accuracy_score(y, pred),
                "f1_macro": f1_score(y, pred, average="macro", zero_division=0)}

    args = TrainingArguments(
        output_dir="bertimbau_intencoes",
        num_train_epochs=6,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        learning_rate=3e-5,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        seed=SEED,
        report_to=[],
    )
    trainer = Trainer(model=modelo_ft, args=args, train_dataset=ds_tr,
                      eval_dataset=ds_va, compute_metrics=metricas_ft)
    trainer.train()

    saida = trainer.predict(ds_te)
    pred_ft = np.array(rotulos)[np.argmax(saida.predictions, axis=-1)]
    m_ft = avaliar(y_teste_int, pred_ft)
    print("\nBERTimbau ajustado — teste:", {k: round(v, 4) for k, v in m_ft.items()})
    print("Modelo clássico  — teste:", {k: round(v, 4) for k, v in m_teste.items()})
    display(pd.DataFrame({f"{MELHOR_REP} + {nome_arquitetura}": m_teste,
                          "BERTimbau fine-tuned": m_ft}).T.round(4))
else:
    print("Fine-tuning desligado. Ative RODAR_FINETUNING = True com GPU para comparar.")

Fine-tuning desligado. Ative RODAR_FINETUNING = True com GPU para comparar.


## 12. Fechamento

**O que este notebook entrega**

- Catálogo de 4 macrointenções × 31 intenções específicas, ancorado no enum `CategoriaIntencao` e nas features priorizadas do `Projeto.md`.
- Dicionário de prompts com centenas de templates do domínio do Metrô, expandido por slots e com ruído de escrita real (sem acento, abreviado, em caixa alta).
- Protocolo de avaliação sem vazamento — split por template, deduplicação, auditoria de quase-duplicatas e a demonstração numérica de quanto o split aleatório infla a métrica.
- Benchmark de TF-IDF, Word2Vec, BERT e combinações híbridas; a execução de referência selecionou **W2V+BERT** com arquitetura hierárquica.
- Artefatos exportados (`artefatos/`) prontos para o serviço de classificação do MVP.

**Limitação honesta.** Os prompts são sintéticos: medem a capacidade de generalizar para *formas de pedir* inéditas, não para *usuários* inéditos. O desempenho de intenção específica ainda é moderado (F1-macro de 0,6777 no teste), e os limiares altos trocam cobertura por segurança. Portanto, W2V+BERT é a melhor alternativa **entre as comparadas nesta rodada**, não uma solução já validada para produção.

**Próximos passos sugeridos**

1. Coletar 20–50 pedidos reais por intenção com o PMO e usá-los como conjunto de teste externo — mantendo os sintéticos apenas no treino.
2. Registrar em produção os casos abaixo do limiar: eles são a fila de rotulação que mais melhora o modelo por hora de trabalho.
3. Acrescentar extração de entidades (`EntidadeExtraida`: projeto, período, critério, valor) — hoje o pipeline resolve *o que* o usuário quer, mas ainda não *sobre o quê*.
4. Reavaliar o catálogo quando as intenções `TRANSACAO` saírem de simulação para execução real: ações com efeito colateral pedem limiar mais alto e confirmação explícita do usuário.